# AIC 2026 - bản tối ưu thời gian BẬT SERVER

Cùng một engine, cùng một API, cùng một kết quả tìm kiếm. Chỉ khác ở khâu chuẩn bị:

| | bản gốc | bản này |
|---|---|---|
| pip | 2 lần, nối tiếp (cell 1 và cell 8) | 1 lần, chạy song song với tải cloudflared |
| cloudflared | tải lúc bật server (nằm trên đường găng) | tải sẵn ở cell 0 |
| file JSON | đọc 1 luồng khi nạp | làm nóng trước ở nền, chồng lên lúc tải model |
| đọc npy | 1 luồng, chờ mạng từng file | 16 luồng đọc trước, giữ nguyên thứ tự |
| chạy lại | nạp lại ~17.000 file nhỏ | đọc 4 file index từ cache |
| copy cục bộ (cell 4b cũ) | đọc mạng + ghi đĩa + đọc lại | bỏ, cache thay thế |

**Logic search không đổi một dòng nào.** Index dựng ra giống hệt bản gốc: cùng danh sách
video theo `sorted()`, cùng thứ tự frame, cùng `normalize_L2`. Cell 4c có `assert` đối chiếu.

Chạy lần lượt từ trên xuống. Lần đầu vẫn tốn thời gian dựng cache; từ lần thứ hai trở đi
(kể cả sau khi restart kernel) cell 4c đọc thẳng cache.

In [ ]:
# ===== CELL 0 - TURBO: cài thư viện + tải cloudflared SONG SONG =====
# Bản gốc chạy nối tiếp: pip faiss/bm25/... (cell 1) -> ... -> pip fastapi + wget
# cloudflared (cell 8). Cả ba đều là I/O mạng và độc lập nhau, không có lý do gì để
# xếp hàng. Gộp về đây, chạy cùng lúc, và tải cloudflared TRƯỚC khi nó nằm trên
# đường găng lúc bật server.
import importlib.util, os, subprocess, sys, time
from concurrent.futures import ThreadPoolExecutor

T_BOOT = time.time()
TIMING = {}


def blog(msg):
    print(f"[{time.time() - T_BOOT:6.1f}s] {msg}", flush=True)


# module dùng để kiểm tra  ->  tên gói pip
PKGS = {
    "faiss": "faiss-cpu",
    "rank_bm25": "rank-bm25",
    "sentence_transformers": "sentence-transformers",
    "google.genai": "google-genai",
    "pydantic": "pydantic",
    "fastapi": "fastapi",
    "uvicorn": "uvicorn",
}

CLOUDFLARED_URL = ("https://github.com/cloudflare/cloudflared/releases/latest"
                   "/download/cloudflared-linux-amd64")


def _missing(mod):
    try:
        return importlib.util.find_spec(mod) is None
    except (ImportError, ValueError, ModuleNotFoundError):
        return True


def job_pip():
    t = time.time()
    need = sorted({pkg for mod, pkg in PKGS.items() if _missing(mod)})
    if not need:
        blog("pip: đã đủ thư viện, bỏ qua")
    else:
        blog(f"pip: cài {' '.join(need)}")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
        blog("pip: xong")
    TIMING["pip"] = time.time() - t


def job_cloudflared():
    # api_server.serve() cũng tự tải nếu thiếu, nhưng lúc đó engine đã nạp xong và
    # mình đang đứng chờ. Tải ở đây thì nó chồng lên lúc pip chạy.
    t = time.time()
    if os.path.exists("./cloudflared"):
        blog("cloudflared: đã có")
    else:
        blog("cloudflared: đang tải...")
        subprocess.run(["wget", "-q", CLOUDFLARED_URL, "-O", "./cloudflared"], check=True)
        os.chmod("./cloudflared", 0o755)
        blog("cloudflared: xong")
    TIMING["cloudflared"] = time.time() - t


with ThreadPoolExecutor(2) as _ex:
    for _f in [_ex.submit(job_pip), _ex.submit(job_cloudflared)]:
        _f.result()   # ném lại lỗi của luồng con, không nuốt

blog("chuẩn bị xong")

In [ ]:
# ===== CELL 1 - trỏ tới dữ liệu + LÀM NÓNG file JSON ở nền =====
# Dữ liệu đã là Kaggle Dataset -> KHÔNG cần snapshot_download, KHÔNG cần HF token.
import glob, os, threading, time
from concurrent.futures import ThreadPoolExecutor

# Đặt tay nếu muốn chắc chắn:
DATABASE_ROOT_PATH = "/kaggle/input/datasets/verse91/json-npy/database"

# Không có thì tự dò: tìm một file *_caption.npy bất kỳ rồi đi ngược lên 3 cấp
# (<root>/Videos_Lxx/<video_id>/<video_id>_caption.npy)
if not os.path.isdir(DATABASE_ROOT_PATH):
    hit = None
    for depth in range(2, 9):
        for root in ("/kaggle/input", "/kaggle/working"):
            found = [f for f in glob.glob(os.path.join(root, *(["*"] * depth), "*_caption.npy"))
                     if "/.cache/" not in f]
            if found:
                hit = found[0]
                break
        if hit:
            break
    if not hit:
        raise FileNotFoundError(
            "Không thấy file *_caption.npy nào dưới /kaggle/input hay /kaggle/working.\n"
            "Đã attach dataset chứa feature BGE-M3 chưa?"
        )
    DATABASE_ROOT_PATH = os.path.dirname(os.path.dirname(os.path.dirname(hit)))

JSON_PATHS = sorted(glob.glob(f"{DATABASE_ROOT_PATH}/*/*/*.json"))
print("DATABASE_ROOT_PATH =", DATABASE_ROOT_PATH)
print(f"số video: {len(JSON_PATHS)}   "
      f"{'✅' if JSON_PATHS else '❌ sai đường dẫn - sửa DATABASE_ROOT_PATH ở trên'}")

# ---------------------------------------------------------------------------
# Làm nóng page cache cho các file JSON, chạy ở NỀN.
#
# /kaggle/input là ổ mạng: nút thắt là ĐỘ TRỄ MỖI FILE chứ không phải băng thông.
# Lượt 1 của bộ nạp đọc toàn bộ *.json bằng một luồng -> hàng nghìn lần chờ nối
# tiếp nhau. Đọc trước bằng 32 luồng NGAY BÂY GIỜ thì khoảng chờ đó chồng lên lúc
# cell 5 tải model BGE-M3 + CLIP (cũng là chờ mạng, nhưng ra HuggingFace).
#
# Chỉ đọc rồi vứt: không giữ gì trong RAM. Cái còn lại là page cache của kernel -
# thu hồi được, và có trần WARM_CAP_GB để không ép bộ nhớ container.
# ---------------------------------------------------------------------------
WARM_JSON = True
WARM_CAP_GB = 6.0
_warm_stat = {}


def _warm_files(paths, cap_bytes, workers=32):
    t, total, lock = time.time(), 0, threading.Lock()

    def rd(p):
        nonlocal total
        with lock:
            if total >= cap_bytes:
                return
        try:
            n = len(open(p, "rb").read())
        except OSError:
            return
        with lock:
            total += n

    with ThreadPoolExecutor(workers) as ex:
        list(ex.map(rd, paths))
    _warm_stat.update(files=len(paths), gb=total / 1e9, sec=time.time() - t)
    print(f"\n🔥 làm nóng xong {len(paths):,} file JSON · {total / 1e9:.2f} GB · "
          f"{time.time() - t:.0f}s", flush=True)


# Đã có cache thì cell 4c không đọc file JSON nào nữa -> làm nóng là đốt băng
# thông vô ích. Dò nông (không glob đệ quy) để khỏi lội 4.000 thư mục video.
HAS_CACHE = any(glob.glob(f"{d}/fingerprint.txt")
                for d in ("/kaggle/input/*", "/kaggle/input/*/*",
                          "/kaggle/input/*/*/*", "/kaggle/working/aic_index_cache"))

if HAS_CACHE:
    print("⚡ thấy cache index -> bỏ qua bước làm nóng, cell 4c sẽ đọc thẳng cache")
elif WARM_JSON and JSON_PATHS:
    WARM_THREAD = threading.Thread(
        target=_warm_files, args=(JSON_PATHS, WARM_CAP_GB * 1e9), daemon=True)
    WARM_THREAD.start()
    print("🔥 đang làm nóng file JSON ở nền - CỨ CHẠY TIẾP các cell dưới")

In [ ]:
# ===== CELL 3 — phân tích prompt bằng LLM =====
# Nguyên văn searcher.py. Khác đúng một chỗ: API key đọc từ Kaggle Secrets thay vì hardcode.
import json
from typing import List, Dict
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

try:
    from kaggle_secrets import UserSecretsClient
    API_KEYS = [UserSecretsClient().get_secret("GOOGLE_API_KEY")]
except Exception:
    API_KEYS = [""]   # dán tay vào đây nếu chưa đặt Secrets

MODELS = [
    "models/gemini-3.5-flash-lite",
    "models/gemma-4-31b-it",
    "models/gemma-4-26b-a4b-it",
]


class ActionQuery(BaseModel):
    spatial_context: List[str] = Field(
        default_factory=list,
        description="List chứa đúng 2 phần tử (1 tiếng Anh, 1 tiếng Việt). CHỈ mô tả HÌNH ẢNH và BỐI CẢNH trực quan nhìn thấy được (ví dụ: 'xe cứu thương', 'cổng xanh')."
    )
    ocr_text: List[str] = Field(
        default_factory=list,
        description="List tiếng Việt các văn bản/chữ cần ocr nằm trên màn hình/frame."
    )
    asr_text: List[str] = Field(
        default_factory=list,
        description="List tiếng Việt chứa nội dung lời nói, thuyết minh hoặc âm thanh (nằm sau cụm 'nói về', 'nhắc đến')."
    )

# Cấu trúc JSON trả về chính
class QueryAnalysisResult(BaseModel):
    reasoning_process: str = Field(
        description="Quy trình suy luận Chain of Thought từng bước: Phân tích cú pháp prompt, xác định Task (1, 2, hay 3), bóc tách bối cảnh, chữ, âm thanh."
    )
    task_type: int = Field(
        description="Loại Task được xác định: 1 (Tìm 1 frame), 2 (VQA), 3 (Chuỗi sự kiện)."
    )
    actions: List[ActionQuery] = Field(
        description="Danh sách các truy vấn. Với Task 1 & 2, mảng này chỉ có 1 phần tử. Với Task 3, mảng này có từ 2 phần tử trở lên, mỗi phần tử là 1 action theo thứ tự thời gian."
    )
    questions: List[str] = Field(
        default_factory=list,
        description="List tiếng Việt trích xuất các câu hỏi cần giải đáp. Chỉ dành cho Task 2, nếu không phải để rỗng []."
    )


# ---------------------------------------------------------------------------
# 3. SYSTEM PROMPT THIẾT KẾ CHO GEMINI / GEMMA
# ---------------------------------------------------------------------------
SYSTEM_PROMPT = """Bạn là Bộ điều phối Phân tích Truy vấn Video Đa phương thức (Multi-Modal Video Query Orchestrator).
Nhiệm vụ của bạn là nhận Prompt từ người dùng, thực hiện suy luận từng bước (Chain-of-Thought) và phân tách các thông tin cần thiết vào cấu trúc JSON định sẵn.

QUY TRÌNH SUY LUẬN TỪNG BƯỚC (Điền vào trường `reasoning_process`):
Bước 1: Phân tích ý định & Xác định loại Task theo CÔNG THỨC SAU:
  - Task 3: Nếu prompt có các từ khóa chỉ thứ tự ("đầu tiên", "sau đó", "tiếp theo", "cuối cùng").
  - Task 2: Nếu prompt KHÔNG CÓ từ khóa thứ tự, NHƯNG CÓ chứa câu hỏi yêu cầu giải đáp (ví dụ: "là ai", "ở đâu", "bao nhiêu", "hãy cho biết").
  - Task 1: Các trường hợp miêu tả cảnh quay thông thường (ví dụ: "Tìm cho tôi cảnh...", "Cho tôi xem cảnh..."). Tuyệt đối không tự bịa ra câu hỏi nếu người dùng không hỏi.

Bước 2: Bóc tách thành các Actions độc lập (Điền vào mảng `actions`).
  - Task 1 & 2: Gộp toàn bộ thông tin vào 1 Action duy nhất.
  - Task 3: Cắt prompt thành các Action riêng biệt theo đúng thứ tự thời gian.

Bước 3: Xử lý dữ liệu bên trong MỖI Action:
  Phân tách rạch ròi 3 luồng thông tin:
  - Lời nói/Âm thanh (nói về Y, nhắc đến Y) -> Đưa vào `asr_text` (Tiếng Việt).
  - Bối cảnh nhìn thấy -> Đưa vào `spatial_context` (Dịch 1 câu Tiếng Anh, 1 câu Tiếng Việt). LƯU Ý QUAN TRỌNG: Toàn bộ đồ vật, UI, đồ họa (ví dụ: "bảng thông số màu cam", "biển báo", "màn hình") phải được coi là HÌNH ẢNH và đưa vào trường này. YÊU CẦU DỊCH THUẬT: Dịch sát nghĩa, trực diện, không thêm bớt ngữ cảnh.
  - Chữ viết trên màn hình -> Đưa vào `ocr_text` (Tiếng Việt). LƯU Ý QUAN TRỌNG: CHỈ đưa vào đây ĐÚNG NỘI DUNG CHỮ viết (ví dụ: "dự án ngàn tỷ"). TUYỆT ĐỐI KHÔNG đưa tên vật thể (như "bảng", "biển", "chữ chạy") vào trường này.

QUY TẮC BẮT BUỘC:
- Nếu là Task 1, trường `questions` BẮT BUỘC phải để rỗng `[]`. Tuyệt đối không tự tạo câu hỏi.
- RẠCH RÒI HÌNH VÀ TIẾNG: Không đưa nội dung lời nói (ASR) vào bối cảnh không gian (Spatial Context).
"""


# ---------------------------------------------------------------------------
# 4. HÀM PHÂN TÍCH PROMPT CHÍNH (CÓ TÍNH NĂNG CƠ CHẾ FALLBACK KEY & MODEL)
# ---------------------------------------------------------------------------
def analyze_prompt(prompt: str) -> dict:
    """
    Thực hiện phân tích prompt sử dụng LLM và trả về dict chứa thông tin được yêu cầu.
    Tự động xoay vòng API Key và Model nếu gặp lỗi quota hoặc kết nối.
    """
    last_error = None

    for key in API_KEYS:
        if not key or key.startswith("YOUR_KEY"):
            continue

        client = genai.Client(api_key=key)

        for model_name in MODELS:
            try:
                # Cấu hình khóa chặt sự sáng tạo, ép mô hình chọn token xác suất cao nhất
                config = types.GenerateContentConfig(
                    system_instruction=SYSTEM_PROMPT,
                    temperature=0.0,  # Ép dùng Greedy Decoding
                    top_k=1,          # Khóa thứ 2: Chỉ nhìn vào 1 token duy nhất
                    top_p=0.1,        # Khóa thứ 3: Cắt bỏ hoàn toàn cái đuôi xác suất
                    response_mime_type="application/json",
                    response_schema=QueryAnalysisResult,
                )

                response = client.models.generate_content(
                    model=model_name,
                    contents=prompt,
                    config=config,
                )

                # Chuyển đổi kết quả JSON trả về thành dict
                result_dict = json.loads(response.text)
                return result_dict

            except Exception as e:
                last_error = e
                print(f"[Warning] Thử nghiệm thất bại với Model {model_name} / Key ...{key[-4:]}: {e}")
                continue

    # Nếu tất cả Key/Model đều thất bại
    raise RuntimeFailureError(f"Tất cả API Key và Model đều không thể xử lý truy vấn. Lỗi cuối cùng: {last_error}")

class RuntimeFailureError(Exception):
    pass


class RuntimeFailureError(Exception):
    pass


# # ---------------------------------------------------------------------------
# # 5. KHỐI THỬ NGHIỆM (MAIN)
# # ---------------------------------------------------------------------------
# if __name__ == "__main__":
#     # Danh sách 3 truy vấn mẫu tương ứng với 3 loại Task
#     test_prompts = [
#         # Test Case 1: Task 1 - Tìm Frame thông thường
#         """tìm cho tôi cảnh bản tin thời sự, người dẫn chương trình đang nói về châu âu đang chống lại nắng nóng""",

#     ]

#     print("=" * 80)
#     print("BẮT ĐẦU KIỂM THỬ HỆ THỐNG PHÂN TÍCH TRUY VẤN DÙNG GEMMA 4")
#     print("=" * 80)

#     for i, test_p in enumerate(test_prompts, 1):
#         print(f"\n--- [TEST CASE {i}] ---")
#         print(f"PROMPT GỐC: {test_p.strip()}\n")

#         try:
#             parsed_result = analyze_prompt(test_p)
#             print("KẾT QUẢ PHÂN TÍCH (JSON STRUCT) :")
#             print(json.dumps(parsed_result, ensure_ascii=False, indent=2))
#         except Exception as err:
#             print(f"Lỗi khi thực thi Test Case {i}: {err}")

#         print("-" * 80)

import os
import json
import numpy as np
import pandas as pd
import faiss
import torch
from typing import Dict, List
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

In [ ]:
# ===== CELL 4 — search engine (nguyên văn searcher.py) =====
import os, json
import numpy as np
import pandas as pd
import faiss, torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from typing import List, Dict


class VideoSearchEngine:
    def __init__(self, db_path: str, bge_model_name: str = 'BAAI/bge-m3', clip_model_name: str = 'clip-ViT-B-32'):
        print("⏳ Khởi tạo Search Engine...")
        self.db_path = db_path
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        print("   🧠 Đang load model BGE-M3 (Text/Speech/OCR/Caption)...")
        self.bge_model = SentenceTransformer(bge_model_name, device=self.device)

        print("   👁️ Đang load model CLIP (Visual)...")
        self.clip_model = SentenceTransformer(clip_model_name, device=self.device)

        self.metadata = []

        self.dim_bge = 1024
        self.dim_clip = 512

        self.index_caption = faiss.IndexFlatIP(self.dim_bge)
        self.index_speech = faiss.IndexFlatIP(self.dim_bge)
        self.index_ocr = faiss.IndexFlatIP(self.dim_bge)
        self.index_clip = faiss.IndexFlatIP(self.dim_clip)

        self.ocr_to_frame_map = []

        self.corpus_caption = []
        self.corpus_speech = []
        self.corpus_ocr = []

        self._load_database()
        self._build_bm25_indices()

    def _load_database(self):
        print(f"📥 Đang load dữ liệu từ Database: {self.db_path} ...")
        caption_vectors, speech_vectors, ocr_vectors, clip_vectors = [], [], [], []
        global_frame_idx = 0

        for folder_name in os.listdir(self.db_path):
            folder_path = os.path.join(self.db_path, folder_name)
            if not os.path.isdir(folder_path): continue

            for video_id in os.listdir(folder_path):
                video_path = os.path.join(folder_path, video_id)
                if not os.path.isdir(video_path): continue

                json_path = os.path.join(video_path, f"{video_id}.json")
                if not os.path.exists(json_path): continue

                with open(json_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)

                try:
                    clip_npy = np.load(os.path.join(video_path, f"{video_id}.npy"))
                    cap_npy = np.load(os.path.join(video_path, f"{video_id}_caption.npy"))
                    speech_npy = np.load(os.path.join(video_path, f"{video_id}_speech.npy"))
                    ocr_npy = np.load(os.path.join(video_path, f"{video_id}_ocr.npy"), allow_pickle=True)
                except Exception as e:
                    print(f"   ⚠️ Lỗi load feature của {video_id}: {e}. Bỏ qua video này.")
                    continue

                for i, frame in enumerate(data['frames']):
                    frame['video_id'] = video_id
                    frame['global_idx'] = global_frame_idx
                    self.metadata.append(frame)

                    self.corpus_caption.append(frame.get('visual_caption', '').lower().split())
                    self.corpus_speech.append(frame.get('speech_text', '').lower().split())
                    self.corpus_ocr.append(frame.get('ocr_text', '').replace('|', ' ').lower().split())

                    clip_vectors.append(clip_npy[i])
                    caption_vectors.append(cap_npy[i])
                    speech_vectors.append(speech_npy[i])

                    frame_ocr_vecs = ocr_npy[i]
                    if frame_ocr_vecs.shape[0] > 0 and frame_ocr_vecs.shape[1] == self.dim_bge:
                        for vec in frame_ocr_vecs:
                            ocr_vectors.append(vec)
                            self.ocr_to_frame_map.append(global_frame_idx)

                    global_frame_idx += 1

        if clip_vectors:
            clip_np = np.vstack(clip_vectors).astype('float32')
            faiss.normalize_L2(clip_np)
            self.index_clip.add(clip_np)

        if caption_vectors:
            cap_np = np.vstack(caption_vectors).astype('float32')
            faiss.normalize_L2(cap_np)
            self.index_caption.add(cap_np)

        if speech_vectors:
            speech_np = np.vstack(speech_vectors).astype('float32')
            faiss.normalize_L2(speech_np)
            self.index_speech.add(speech_np)

        if ocr_vectors:
            ocr_np = np.vstack(ocr_vectors).astype('float32')
            faiss.normalize_L2(ocr_np)
            self.index_ocr.add(ocr_np)

        print(f"✅ Hoàn tất load {len(self.metadata)} frames vào hệ thống!")

    def _build_bm25_indices(self):
        print("🧠 Đang khởi tạo thuật toán BM25...")
        self.bm25_caption = BM25Okapi(self.corpus_caption)
        self.bm25_speech = BM25Okapi(self.corpus_speech)
        self.bm25_ocr = BM25Okapi(self.corpus_ocr)

    def _min_max_scale(self, scores: np.ndarray) -> np.ndarray:
        if len(scores) == 0 or np.max(scores) == np.min(scores):
            return np.zeros_like(scores, dtype=float)
        return (scores - np.min(scores)) / (np.max(scores) - np.min(scores))

    # =====================================================================
    # HÀM SEARCH THÔNG THƯỜNG (CHO TASK 1 VÀ 2)
    # =====================================================================
    def search(self, action_query: Dict, top_k: int = 10, visualize: bool = False, weights: Dict[str, float] = None):
        if weights is None:
            weights = {'visual': 1.0, 'speech': 1.0, 'ocr': 1.0}

        num_frames = len(self.metadata)

        scores_dict = {
            'faiss_clip': np.zeros(num_frames), 'faiss_caption': np.zeros(num_frames), 'bm25_caption': np.zeros(num_frames),
            'faiss_speech': np.zeros(num_frames), 'bm25_speech': np.zeros(num_frames),
            'faiss_ocr': np.zeros(num_frames), 'bm25_ocr': np.zeros(num_frames)
        }

        active_modalities = []

        if action_query.get('spatial_context') and len(action_query['spatial_context']) > 0:
            active_modalities.append('visual')
            context_text_eng = action_query['spatial_context'][0]

            clip_query_emb = self.clip_model.encode([context_text_eng], normalize_embeddings=True).astype('float32')
            D_clip, I_clip = self.index_clip.search(clip_query_emb, num_frames)
            for dist, idx in zip(D_clip[0], I_clip[0]): scores_dict['faiss_clip'][idx] = dist

            bge_query_emb = self.bge_model.encode([context_text_eng], normalize_embeddings=True).astype('float32')
            D_cap, I_cap = self.index_caption.search(bge_query_emb, num_frames)
            for dist, idx in zip(D_cap[0], I_cap[0]): scores_dict['faiss_caption'][idx] = dist

            scores_dict['bm25_caption'] = self.bm25_caption.get_scores(context_text_eng.lower().split())

        if action_query.get('asr_text') and len(action_query['asr_text']) > 0:
            active_modalities.append('speech')
            asr_text = " ".join(action_query['asr_text'])

            query_emb = self.bge_model.encode([asr_text], normalize_embeddings=True).astype('float32')
            D, I = self.index_speech.search(query_emb, num_frames)
            for dist, idx in zip(D[0], I[0]): scores_dict['faiss_speech'][idx] = dist

            scores_dict['bm25_speech'] = self.bm25_speech.get_scores(asr_text.lower().split())

        if action_query.get('ocr_text') and len(action_query['ocr_text']) > 0:
            active_modalities.append('ocr')
            ocr_text = " ".join(action_query['ocr_text'])

            query_emb = self.bge_model.encode([ocr_text], normalize_embeddings=True).astype('float32')
            D, I = self.index_ocr.search(query_emb, self.index_ocr.ntotal)
            for dist, ocr_idx in zip(D[0], I[0]):
                frame_idx = self.ocr_to_frame_map[ocr_idx]
                if dist > scores_dict['faiss_ocr'][frame_idx]:
                    scores_dict['faiss_ocr'][frame_idx] = dist

            scores_dict['bm25_ocr'] = self.bm25_ocr.get_scores(ocr_text.lower().split())

        active_weight_sum = sum([weights.get(m, 1.0) for m in active_modalities])

        if active_weight_sum > 0:
            w_vis = weights.get('visual', 1.0) / active_weight_sum if 'visual' in active_modalities else 0.0
            w_speech = weights.get('speech', 1.0) / active_weight_sum if 'speech' in active_modalities else 0.0
            w_ocr = weights.get('ocr', 1.0) / active_weight_sum if 'ocr' in active_modalities else 0.0
        else:
            w_vis = w_speech = w_ocr = 0.0

        normalized_scores = {}
        for key, scores in scores_dict.items():
            normalized_scores[key] = self._min_max_scale(scores)

        score_vis = (normalized_scores['faiss_clip'] + normalized_scores['faiss_caption'] + normalized_scores['bm25_caption']) / 3.0
        score_speech = (normalized_scores['faiss_speech'] + normalized_scores['bm25_speech']) / 2.0
        score_ocr = (normalized_scores['faiss_ocr'] + normalized_scores['bm25_ocr']) / 2.0

        total_scores = (w_vis * score_vis) + (w_speech * score_speech) + (w_ocr * score_ocr)

        top_indices = np.argsort(total_scores)[::-1][:top_k]
        results, final_return_ids = [], []

        for idx in top_indices:
            meta = self.metadata[idx]

            # Gói ID vào list để đồng nhất với Task 3
            final_return_ids.append([f"{meta['video_id']}_{meta['original_frame_idx']}"])

            if visualize:
                results.append({
                    "Video_ID": meta['video_id'],
                    "Orig_ID": f"[{meta['original_frame_idx']}]",
                    "Total(0-1)": round(total_scores[idx], 4),
                    "F_CLIP(Vis)": round(normalized_scores['faiss_clip'][idx], 3),
                    "F_BGE(Cap)": round(normalized_scores['faiss_caption'][idx], 3),
                    "BM25(Cap)": round(normalized_scores['bm25_caption'][idx], 3),
                    "F_BGE(Spch)": round(normalized_scores['faiss_speech'][idx], 3),
                    "BM25(Spch)": round(normalized_scores['bm25_speech'][idx], 3),
                    "F_BGE(OCR)": round(normalized_scores['faiss_ocr'][idx], 3),
                    "BM25(OCR)": round(normalized_scores['bm25_ocr'][idx], 3)
                })

        if visualize:
            df = pd.DataFrame(results)
            print(f"\n📊 TRỌNG SỐ THỰC TẾ (Tổng=1.0): Visual: {w_vis:.2f} | Speech: {w_speech:.2f} | OCR: {w_ocr:.2f}")
            print(df.to_string(index=False))


        # Điểm để web hiện thẳng lên thumbnail. Cất vào self chứ KHÔNG đổi kiểu trả
        # về, để mọi chỗ đang gọi search() khỏi phải sửa theo.
        #
        # CẢNH BÁO khi đọc con số này: mỗi mảng điểm thành phần đã bị min-max hoá
        # THEO TỪNG TRUY VẤN. Nên nó chỉ so sánh được giữa các frame trong CÙNG một
        # lần tìm. Một câu tìm trúng và một câu tìm trượt hoàn toàn vẫn cho điểm đỉnh
        # xấp xỉ nhau - đừng dùng nó để đoán "câu này có tìm ra không".
        # Phần đáng tin là PHÂN RÃ: ô này lên cao nhờ hình ảnh hay nhờ OCR.
        self.last_scores = {
            f"{self.metadata[i]['video_id']}_{self.metadata[i]['original_frame_idx']}": {
                "total": round(float(total_scores[i]), 4),
                "clip": round(float(normalized_scores['faiss_clip'][i]), 3),
                "cap_bge": round(float(normalized_scores['faiss_caption'][i]), 3),
                "cap_bm25": round(float(normalized_scores['bm25_caption'][i]), 3),
                "asr_bge": round(float(normalized_scores['faiss_speech'][i]), 3),
                "asr_bm25": round(float(normalized_scores['bm25_speech'][i]), 3),
                "ocr_bge": round(float(normalized_scores['faiss_ocr'][i]), 3),
                "ocr_bm25": round(float(normalized_scores['bm25_ocr'][i]), 3),
            }
            for i in top_indices
        }
        return final_return_ids

    # =====================================================================
    # HÀM TÌM KIẾM CHUỖI SỰ KIỆN QUY HOẠCH ĐỘNG (CHO TASK 3)
    # =====================================================================
    def search_temporal_events(self, actions: List[Dict], top_k: int = 5, visualize: bool = False, weights: Dict[str, float] = None):
        """
        Tìm kiếm chuỗi sự kiện, lấy RA TẤT CẢ TỔ HỢP TỐT (NMS) thay vì 1 tổ hợp/video.
        """
        if weights is None:
            weights = {'visual': 1.0, 'speech': 1.0, 'ocr': 1.0}

        num_frames = len(self.metadata)
        num_events = len(actions)

        if num_events == 0:
            return []

        all_events_scores = []
        all_detailed_scores = []

        # --- BƯỚC 1: TÍNH ĐIỂM CHO TỪNG ACTION ĐỘC LẬP ---
        for action in actions:
            scores_dict = {
                'faiss_clip': np.zeros(num_frames), 'faiss_caption': np.zeros(num_frames), 'bm25_caption': np.zeros(num_frames),
                'faiss_speech': np.zeros(num_frames), 'bm25_speech': np.zeros(num_frames),
                'faiss_ocr': np.zeros(num_frames), 'bm25_ocr': np.zeros(num_frames)
            }

            active_modalities = []

            # Visual
            if action.get('spatial_context') and len(action['spatial_context']) > 0:
                active_modalities.append('visual')
                eng_text = action['spatial_context'][0]

                clip_emb = self.clip_model.encode([eng_text], normalize_embeddings=True).astype('float32')
                D_clip, I_clip = self.index_clip.search(clip_emb, num_frames)
                for dist, idx in zip(D_clip[0], I_clip[0]): scores_dict['faiss_clip'][idx] = dist

                bge_emb_eng = self.bge_model.encode([eng_text], normalize_embeddings=True).astype('float32')
                D_cap, I_cap = self.index_caption.search(bge_emb_eng, num_frames)
                for dist, idx in zip(D_cap[0], I_cap[0]): scores_dict['faiss_caption'][idx] = dist
                scores_dict['bm25_caption'] = self.bm25_caption.get_scores(eng_text.lower().split())

            # Speech
            if action.get('asr_text') and len(action['asr_text']) > 0:
                active_modalities.append('speech')
                vi_text = " ".join(action['asr_text'])

                bge_emb_vi = self.bge_model.encode([vi_text], normalize_embeddings=True).astype('float32')
                D_spch, I_spch = self.index_speech.search(bge_emb_vi, num_frames)
                for dist, idx in zip(D_spch[0], I_spch[0]): scores_dict['faiss_speech'][idx] = dist
                scores_dict['bm25_speech'] = self.bm25_speech.get_scores(vi_text.lower().split())

            # OCR
            if action.get('ocr_text') and len(action['ocr_text']) > 0:
                active_modalities.append('ocr')
                vi_text_ocr = " ".join(action['ocr_text'])

                bge_emb_vi_ocr = self.bge_model.encode([vi_text_ocr], normalize_embeddings=True).astype('float32')
                D_ocr, I_ocr = self.index_ocr.search(bge_emb_vi_ocr, self.index_ocr.ntotal)
                for dist, ocr_idx in zip(D_ocr[0], I_ocr[0]):
                    frame_idx = self.ocr_to_frame_map[ocr_idx]
                    if dist > scores_dict['faiss_ocr'][frame_idx]:
                        scores_dict['faiss_ocr'][frame_idx] = dist
                scores_dict['bm25_ocr'] = self.bm25_ocr.get_scores(vi_text_ocr.lower().split())

            # Chuẩn hóa & Trọng số
            active_weight_sum = sum([weights.get(m, 1.0) for m in active_modalities])
            if active_weight_sum > 0:
                w_vis = weights.get('visual', 1.0) / active_weight_sum if 'visual' in active_modalities else 0.0
                w_speech = weights.get('speech', 1.0) / active_weight_sum if 'speech' in active_modalities else 0.0
                w_ocr = weights.get('ocr', 1.0) / active_weight_sum if 'ocr' in active_modalities else 0.0
            else:
                w_vis = w_speech = w_ocr = 0.0

            normalized_scores = {k: self._min_max_scale(v) for k, v in scores_dict.items()}
            all_detailed_scores.append(normalized_scores)

            score_vis = (normalized_scores['faiss_clip'] + normalized_scores['faiss_caption'] + normalized_scores['bm25_caption']) / 3.0
            score_speech = (normalized_scores['faiss_speech'] + normalized_scores['bm25_speech']) / 2.0
            score_ocr = (normalized_scores['faiss_ocr'] + normalized_scores['bm25_ocr']) / 2.0

            event_total_score = (w_vis * score_vis) + (w_speech * score_speech) + (w_ocr * score_ocr)
            all_events_scores.append(event_total_score)

        # --- BƯỚC 2: GOM NHÓM FRAMES ---
        video_to_frames = {}
        for idx, meta in enumerate(self.metadata):
            vid = meta['video_id']
            if vid not in video_to_frames:
                video_to_frames[vid] = []
            video_to_frames[vid].append(idx)

        # --- BƯỚC 3: QUY HOẠCH ĐỘNG VÀ LẤY NHIỀU CHUỖI ---
        global_sequences = [] # Chứa tất cả các chuỗi hợp lệ từ tất cả các video

        for vid, frame_indices in video_to_frames.items():
            n_frames = len(frame_indices)
            if n_frames < num_events:
                continue

            dp = np.full((num_events, n_frames), -1.0)
            trace = np.full((num_events, n_frames), -1, dtype=int)

            # Khởi tạo dp cho sự kiện đầu tiên
            for f in range(n_frames):
                global_f = frame_indices[f]
                dp[0][f] = all_events_scores[0][global_f]

            # Chạy DP
            for e in range(1, num_events):
                running_max_score = -1.0
                best_prev_f = -1

                for f in range(1, n_frames):
                    if dp[e-1][f-1] > running_max_score:
                        running_max_score = dp[e-1][f-1]
                        best_prev_f = f - 1

                    if running_max_score > -1.0:
                        global_f = frame_indices[f]
                        dp[e][f] = running_max_score + all_events_scores[e][global_f]
                        trace[e][f] = best_prev_f

            # ---- LẤY RA CÁC ĐỈNH (LOCAL MAXIMA) BẰNG NMS ----
            end_scores = dp[num_events-1]
            sorted_f_indices = np.argsort(end_scores)[::-1] # Sắp xếp frame kết thúc từ cao xuống thấp

            chosen_ends = []
            min_frame_gap = 5 # Khoảng cách tối thiểu (số frame index) để tránh lấy 2 chuỗi trùng nhau

            for f in sorted_f_indices:
                score = end_scores[f]
                # Bỏ qua nếu điểm âm hoặc bằng 0
                if score <= 0:
                    break

                # NMS: Nếu frame kết thúc này quá gần với một chuỗi đã chọn, bỏ qua nó
                if any(abs(f - chosen_f) < min_frame_gap for chosen_f in chosen_ends):
                    continue

                chosen_ends.append(f)

                # Backtrack để lấy toàn bộ ID của chuỗi
                seq_global_indices = []
                curr_f = f
                for e in range(num_events-1, -1, -1):
                    seq_global_indices.append(frame_indices[curr_f])
                    curr_f = trace[e][curr_f]

                seq_global_indices.reverse()

                global_sequences.append({
                    'video_id': vid,
                    'avg_score': score / num_events,
                    'sequence_indices': seq_global_indices
                })

                # Tùy chọn: Chỉ lấy tối đa 10 chuỗi tốt nhất từ 1 video
                if len(chosen_ends) >= 10:
                    break

        # --- BƯỚC 4: XẾP HẠNG GLOBAL VÀ HIỂN THỊ ---
        # Sắp xếp toàn bộ chuỗi thu được từ TẤT CẢ video theo điểm trung bình giảm dần
        global_sequences.sort(key=lambda x: x['avg_score'], reverse=True)
        top_sequences = global_sequences[:top_k]

        final_return_ids = []
        results = []
        # TRAKE chấm điểm theo CHUỖI chứ không theo từng frame, nên mọi frame
        # trong một chuỗi dùng chung điểm trung bình của chuỗi đó. Gán lại từ đầu
        # để điểm của lần tìm KIS trước không sót lại.
        self.last_scores = {}

        for res in top_sequences:
            vid = res['video_id']
            avg_score = round(res['avg_score'], 4)
            seq_global_indices = res['sequence_indices']

            seq_meta = [self.metadata[idx] for idx in seq_global_indices]
            orig_ids_list = [m['original_frame_idx'] for m in seq_meta]
            orig_ids_str = "[" + ", ".join(map(str, orig_ids_list)) + "]"

            final_return_ids.append([f"{vid}_{frame_id}" for frame_id in orig_ids_list])
            for _fid in orig_ids_list:
                self.last_scores[f"{vid}_{_fid}"] = {"total": avg_score}

            if visualize:
                avg_details = {k: 0.0 for k in all_detailed_scores[0].keys()}
                for e_idx, global_f in enumerate(seq_global_indices):
                    for k in avg_details.keys():
                        avg_details[k] += all_detailed_scores[e_idx][k][global_f]

                for k in avg_details.keys():
                    avg_details[k] /= num_events

                results.append({
                    "Video_ID": vid,
                    "Orig_ID": orig_ids_str,
                    "Total(0-1)": avg_score,
                    "F_CLIP(Vis)": round(avg_details['faiss_clip'], 3),
                    "F_BGE(Cap)": round(avg_details['faiss_caption'], 3),
                    "BM25(Cap)": round(avg_details['bm25_caption'], 3),
                    "F_BGE(Spch)": round(avg_details['faiss_speech'], 3),
                    "BM25(Spch)": round(avg_details['bm25_speech'], 3),
                    "F_BGE(OCR)": round(avg_details['faiss_ocr'], 3),
                    "BM25(OCR)": round(avg_details['bm25_ocr'], 3)
                })

        if visualize:
            df = pd.DataFrame(results)
            print(f"\n📊 KẾT QUẢ TÌM KIẾM CHUỖI SỰ KIỆN (TOP {len(top_sequences)} CHUỖI TRÊN TOÀN DB):")
            print(df.to_string(index=False))

        return final_return_ids

In [ ]:
# ===== CELL 4d — tầng trả kết quả theo task (nguyên văn searcher.py) =====
# Trả về list dict: {"loại task": "task N", "kết quả": [chuỗi đúng định dạng BTC]}
#   task 1: "video_id, frame_id"
#   task 2: "video_id, frame_id, question"
#   task 3: "video_id, frame_id1, frame_id2, ..., frame_idn"


def process_queries(queries: List[Dict], db_path: str = None, top_k: int = 5,
                    visualize: bool = False, engine=None) -> List[Dict]:
    """
    Hàm xử lý danh sách các truy vấn, tự động định tuyến, in log trực quan (nếu bật) và format kết quả.
    - top_k: Số lượng kết quả tốt nhất trả về cho mỗi query.
    """
    # 1. Khởi tạo Search Engine
    # Dùng lại engine đã tạo ở cell 5. Tạo engine mới sẽ nạp lại 4.2GB -> OOM.
    if engine is None:
        engine = VideoSearchEngine(db_path=db_path)
    final_output = []

    # 2. Xử lý từng query
    for i, q in enumerate(queries, 1):
        user_prompt = q.get("prompt", "")
        custom_weights = q.get("weights", {'visual': 1.0, 'speech': 1.0, 'ocr': 1.0})

        # --- BỔ SUNG LOGGING GIAO DIỆN (CHỈ KHI VISUALIZE = TRUE) ---
        if visualize:
            print("\n\n" + "="*80)
            print(f"🚀 [TEST CASE {i}]")
            print(f"👤 NGƯỜI DÙNG NHẬP: {user_prompt}")
            print(f"⚖️ TRỌNG SỐ ÁP DỤNG: Visual: {custom_weights.get('visual')}, Speech: {custom_weights.get('speech')}, OCR: {custom_weights.get('ocr')}")
            print(f"🎯 TRẢ VỀ: Top {top_k} kết quả")
            print("="*80)
            print("\n🧠 Đang gọi LLM (Gemini/Gemma) để phân tích truy vấn...")

        # 3. GỌI LLM ĐỂ PHÂN TÍCH PROMPT
        try:
            parsed_query = analyze_prompt(user_prompt)
            if visualize:
                print("\n✅ KẾT QUẢ LLM PHÂN TÍCH (JSON):")
                print(json.dumps(parsed_query, ensure_ascii=False, indent=2))
        except Exception as e:
            if visualize:
                print(f"❌ Lỗi khi phân tích prompt: {e}")
            parsed_query = None

        if not parsed_query:
            continue

        task_type = parsed_query.get('task_type', 1)
        actions = parsed_query.get('actions', [])
        questions = parsed_query.get('questions', [])

        formatted_results = []

        if visualize:
            print("\n🔍 ĐANG TIẾN HÀNH TÌM KIẾM TRONG DATABASE...")

        # --- ĐỊNH TUYẾN VÀ LẤY KẾT QUẢ THÔ ---
        if task_type == 3 and len(actions) > 1:
            if visualize:
                print("⚡ Phát hiện truy vấn Chuỗi Sự Kiện (Task 3). Kích hoạt Dynamic Programming...")
            raw_results = engine.search_temporal_events(
                actions=actions,
                top_k=top_k,
                visualize=visualize,
                weights=custom_weights
            )

            # Format Task 3: <video_id>, <frame_id1>, ..., <frame_idn>
            for res in raw_results:
                if not res: continue
                # res có dạng: ['L23_V014_1250', 'L23_V014_1625', ...]
                vid = res[0].rsplit('_', 1)[0]
                frame_ids = [item.rsplit('_', 1)[1] for item in res]

                formatted_string = f"{vid}, " + ", ".join(frame_ids)
                formatted_results.append(formatted_string)

        else:
            if visualize:
                print("⚡ Phát hiện truy vấn Frame đơn lẻ (Task 1/2). Kích hoạt Late Fusion Retrieval...")
            single_action = actions[0] if len(actions) > 0 else {}
            raw_results = engine.search(
                action_query=single_action,
                top_k=top_k,
                visualize=visualize,
                weights=custom_weights
            )

            # Format Task 1 & 2
            for res in raw_results:
                if not res: continue
                # res có dạng: ['L21_V012_3691']
                vid = res[0].rsplit('_', 1)[0]
                frame_id = res[0].rsplit('_', 1)[1]

                if task_type == 2:
                    # Task 2: <video_id>, <frame_id>, <answer>
                    answer = questions[0] if len(questions) > 0 else "Unknown Question"
                    formatted_string = f"{vid}, {frame_id}, {answer}"
                else:
                    # Task 1: <video_id>, <frame_id>
                    formatted_string = f"{vid}, {frame_id}"

                formatted_results.append(formatted_string)

        if visualize:
            print(f"\n🎯 Danh sách {top_k} chuỗi kết quả đã format (Để submit/hiển thị):")
            print(formatted_results)

        # --- LƯU DICT KẾT QUẢ CHO TASK ---
        final_output.append({
            "loại task": f"task {task_type}",
            "kết quả": formatted_results
        })

    return final_output

# ---- chạy thử: TRUYỀN engine đã có, đừng để nó tự tạo ----
# FINAL = process_queries(
#     [{"prompt": PROMPT, "weights": WEIGHTS}],
#     top_k=100, visualize=False, engine=engine,
# )
# print(json.dumps(FINAL, ensure_ascii=False, indent=4))

In [ ]:
# ===== CELL 4c - nạp dữ liệu: CACHE ĐĨA + ĐỌC SONG SONG =====
# Ghi đè _load_database. Index ra GIỐNG HỆT bản gốc: cùng danh sách video theo
# sorted(), cùng thứ tự frame, cùng normalize_L2 - có assert đối chiếu ở cuối.
#
# Hai thay đổi, cả hai đều nằm NGOÀI phần tính toán:
#
#   1. CACHE. Dựng xong thì ghi 4 index FAISS + metadata ra đĩa. Lần chạy sau đọc
#      lại 4 file lớn thay vì ~17.000 file nhỏ qua ổ mạng. Đây là chỗ ăn tiền
#      nhất: cùng số byte, nhưng ít hơn 4.000 lần số lần chờ mạng.
#
#   2. ĐỌC TRƯỚC. Mỗi lượt duyệt video có WORKERS luồng đọc trước AHEAD video,
#      thứ tự XỬ LÝ giữ nguyên. Che độ trễ mạng, trần bộ nhớ vẫn cố định vì không
#      bao giờ giữ quá AHEAD mảng cùng lúc.
#
# Vì sao KHÔNG chia khối gọi add() nhiều lần: IndexFlatIP lưu trong std::vector
# C++, vượt sức chứa là cấp phát GẤP ĐÔI rồi copy -> gọi add() 15 lần còn tốn hơn
# 1 lần. Cách đúng vẫn là cấp phát sẵn mảng đích một lần rồi add() đúng một lần.
import gc, glob, hashlib, itertools, json, os, pickle, shutil, time
from collections import deque
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import faiss

WORKERS = 16          # luồng đọc file
AHEAD = 8             # đọc trước bao nhiêu video
CACHE_WRITE = "/kaggle/working/aic_index_cache"
CACHE_FILES = ("clip.faiss", "caption.faiss", "speech.faiss", "ocr.faiss",
               "metadata.pkl", "ocr_map.npy", "fingerprint.txt")


def _mem_gb():
    for p in ("/sys/fs/cgroup/memory.stat", "/sys/fs/cgroup/memory/memory.stat"):
        try:
            st = dict(l.split()[:2] for l in open(p))
            return int(st.get("anon", 0)) / 1e9
        except OSError:
            pass
    return -1.0


def _log(tag):
    print(f"   [{_mem_gb():5.1f} GB] {tag}", flush=True)


def _fingerprint(db_path):
    """Vân tay của bộ dữ liệu. Cache chỉ dùng lại khi khớp chính xác -> đổi dataset
    là cache tự bị bỏ, không bao giờ phục vụ index của bộ dữ liệu khác."""
    vids = sorted(os.path.basename(os.path.dirname(p))
                  for p in glob.glob(f"{db_path}/*/*/*.json"))
    return f"{len(vids)}v-{hashlib.sha1(chr(10).join(vids).encode()).hexdigest()[:16]}"


def _cache_dirs():
    """Nơi có thể chứa cache. Ưu tiên dataset đã attach (còn sau khi đổi phiên),
    sau đó tới /kaggle/working (chỉ còn trong phiên hiện tại).
    Cố ý KHÔNG dùng glob recursive: nó sẽ lội hết 4.000 thư mục video trên ổ mạng."""
    found = []
    for d in ("/kaggle/input/*", "/kaggle/input/*/*", "/kaggle/input/*/*/*"):
        found += [os.path.dirname(p) for p in glob.glob(f"{d}/clip.faiss")]
    return found + [CACHE_WRITE]


def _cache_load(self, fp):
    for d in _cache_dirs():
        if not all(os.path.exists(os.path.join(d, f)) for f in CACHE_FILES):
            continue
        got = open(os.path.join(d, "fingerprint.txt")).read().strip()
        if got != fp:
            print(f"   ⚠️  cache {d} là của bộ dữ liệu khác ({got}) -> bỏ qua")
            continue

        t = time.time()
        print(f"⚡ nạp từ cache: {d}")
        self.index_clip = faiss.read_index(os.path.join(d, "clip.faiss"))
        self.index_caption = faiss.read_index(os.path.join(d, "caption.faiss"))
        self.index_speech = faiss.read_index(os.path.join(d, "speech.faiss"))
        self.index_ocr = faiss.read_index(os.path.join(d, "ocr.faiss"))
        with open(os.path.join(d, "metadata.pkl"), "rb") as fh:
            self.metadata = pickle.load(fh)
        self.ocr_to_frame_map = np.load(os.path.join(d, "ocr_map.npy")).tolist()

        # corpus BM25 dựng lại từ metadata - NGUYÊN VĂN ba dòng của bản gốc,
        # nên rẻ hơn ghi corpus ra đĩa mà kết quả không lệch một token nào.
        self.corpus_caption = [m.get("visual_caption", "").lower().split()
                               for m in self.metadata]
        self.corpus_speech = [m.get("speech_text", "").lower().split()
                              for m in self.metadata]
        self.corpus_ocr = [m.get("ocr_text", "").replace("|", " ").lower().split()
                           for m in self.metadata]

        assert self.index_clip.d == self.dim_clip, "cache sai chiều CLIP"
        assert self.index_caption.d == self.dim_bge, "cache sai chiều BGE"
        assert self.index_clip.ntotal == len(self.metadata), "cache lệch số frame"
        assert self.index_caption.ntotal == len(self.metadata), "caption lệch số frame"
        assert self.index_speech.ntotal == len(self.metadata), "speech lệch số frame"
        assert self.index_ocr.ntotal == len(self.ocr_to_frame_map), "ocr lệch map"
        _log(f"cache xong: {len(self.metadata):,} frame · {time.time() - t:.0f}s")
        return True
    return False


def _cache_save(self, fp, out=CACHE_WRITE):
    need = (self.index_clip.ntotal * self.dim_clip
            + (self.index_caption.ntotal + self.index_speech.ntotal
               + self.index_ocr.ntotal) * self.dim_bge) * 4 / 1e9
    free = shutil.disk_usage(os.path.dirname(out)).free / 1e9
    if free < need + 2:
        print(f"⚠️  cần ~{need:.1f} GB để ghi cache, chỉ còn {free:.1f} GB -> bỏ qua")
        return
    os.makedirs(out, exist_ok=True)
    t = time.time()
    faiss.write_index(self.index_clip, os.path.join(out, "clip.faiss"))
    faiss.write_index(self.index_caption, os.path.join(out, "caption.faiss"))
    faiss.write_index(self.index_speech, os.path.join(out, "speech.faiss"))
    faiss.write_index(self.index_ocr, os.path.join(out, "ocr.faiss"))
    with open(os.path.join(out, "metadata.pkl"), "wb") as fh:
        pickle.dump(self.metadata, fh, protocol=pickle.HIGHEST_PROTOCOL)
    np.save(os.path.join(out, "ocr_map.npy"),
            np.asarray(self.ocr_to_frame_map, dtype=np.int64))
    # fingerprint ghi SAU CÙNG: cache dở dang thì thiếu file này -> không được dùng.
    with open(os.path.join(out, "fingerprint.txt"), "w") as fh:
        fh.write(fp)
    gb = sum(os.path.getsize(os.path.join(out, f)) for f in CACHE_FILES) / 1e9
    print(f"💾 đã ghi cache {out} · {gb:.1f} GB · {time.time() - t:.0f}s")
    print("   Lần chạy sau (kể cả sau restart kernel) cell này sẽ đọc thẳng cache.")
    print("   Muốn giữ qua PHIÊN KHÁC: Save Version -> Output, rồi attach thư mục")
    print("   này làm dataset cho notebook.")


def _prefetch(items, loader, workers=WORKERS, ahead=AHEAD):
    """Duyệt `items` ĐÚNG THỨ TỰ nhưng đọc trước `ahead` phần tử bằng `workers`
    luồng. Thứ tự yield không đổi -> thứ tự vector trong index không đổi."""
    it = iter(items)
    q = deque()
    with ThreadPoolExecutor(workers) as ex:
        for x in itertools.islice(it, ahead):
            q.append((x, ex.submit(loader, x)))
        while q:
            x, fut = q.popleft()
            nxt = next(it, None)
            if nxt is not None:
                q.append((nxt, ex.submit(loader, nxt)))
            yield x, fut.result()


def _load_database_fast(self):
    fp = _fingerprint(self.db_path)
    if _cache_load(self, fp):
        print(f"✅ Hoàn tất load {len(self.metadata)} frames vào hệ thống!")
        return

    print(f"📥 Đang load: {self.db_path}  (chưa có cache - dựng lần đầu)")
    _log("bắt đầu")

    # ---- liệt kê video: giữ nguyên sorted() của bản gốc để thứ tự không đổi ----
    cands = []
    for folder in sorted(os.listdir(self.db_path)):
        fpth = os.path.join(self.db_path, folder)
        if not os.path.isdir(fpth):
            continue
        for vid in sorted(os.listdir(fpth)):
            vp = os.path.join(fpth, vid)
            jp = os.path.join(vp, f"{vid}.json")
            if not (os.path.isdir(vp) and os.path.exists(jp)):
                continue
            if not all(os.path.exists(os.path.join(vp, f"{vid}{s}.npy"))
                       for s in ("", "_caption", "_speech", "_ocr")):
                print(f"   ⚠️ {vid}: thiếu npy, bỏ qua")
                continue
            cands.append((vp, vid, jp))

    # ---- LƯỢT 1: đọc json, đếm số vector, dựng metadata + corpus ----
    def _read_meta(c):
        vp, vid, jp = c
        with open(jp, encoding="utf-8") as f:
            frames = json.load(f)["frames"]
        ocr = np.load(os.path.join(vp, f"{vid}_ocr.npy"), allow_pickle=True)
        return frames, ocr

    videos = []
    n_frames = n_ocr = 0
    for (vp, vid, jp), (frames, ocr_npy) in _prefetch(cands, _read_meta):
        counts = []
        for i, fr in enumerate(frames):
            fr["video_id"] = vid
            fr["global_idx"] = n_frames
            self.metadata.append(fr)
            self.corpus_caption.append(fr.get("visual_caption", "").lower().split())
            self.corpus_speech.append(fr.get("speech_text", "").lower().split())
            self.corpus_ocr.append(fr.get("ocr_text", "").replace("|", " ").lower().split())

            v = ocr_npy[i] if i < len(ocr_npy) else None
            k = v.shape[0] if getattr(v, "ndim", 0) == 2 and v.shape[1] == self.dim_bge else 0
            counts.append(k)
            for _ in range(k):
                self.ocr_to_frame_map.append(n_frames)
            n_ocr += k
            n_frames += 1
        videos.append((vp, vid, len(frames), counts))
        del ocr_npy
    gc.collect()
    _log(f"lượt 1 xong: {n_frames:,} frame · {n_ocr:,} vector OCR · {len(videos)} video")

    # ---- LƯỢT 2: cấp phát SẴN từng ma trận, đổ vào, add MỘT lần ----
    def build(index, dim, n, fill, name):
        m = np.empty((n, dim), dtype="float32")   # cấp phát đúng một lần
        fill(m)
        faiss.normalize_L2(m)                      # chuẩn hoá theo từng dòng
        index.add(m)
        del m
        gc.collect()
        _log(f"{name}: {index.ntotal:,} vector")

    def filler(suffix, per_frame=False):
        def f(m):
            r = 0

            def _rd(v):
                vp, vid, nf, counts = v
                return np.load(os.path.join(vp, f"{vid}{suffix}.npy"),
                               allow_pickle=per_frame)

            for (vp, vid, nf, counts), a in _prefetch(videos, _rd):
                if per_frame:
                    for i in range(nf):
                        k = counts[i]
                        if k:
                            m[r:r + k] = a[i]
                            r += k
                else:
                    m[r:r + nf] = a[:nf]
                    r += nf
                del a
            assert r == len(m), f"đổ {r} dòng nhưng cấp phát {len(m)}"
        return f

    build(self.index_clip, self.dim_clip, n_frames, filler(""), "clip")
    build(self.index_caption, self.dim_bge, n_frames, filler("_caption"), "caption")
    build(self.index_speech, self.dim_bge, n_frames, filler("_speech"), "speech")
    build(self.index_ocr, self.dim_bge, n_ocr, filler("_ocr", True), "ocr")

    assert self.index_ocr.ntotal == len(self.ocr_to_frame_map), "index_ocr lệch ocr_to_frame_map"
    assert self.index_clip.ntotal == len(self.metadata), "index_clip lệch metadata"
    print(f"✅ Hoàn tất load {len(self.metadata)} frames vào hệ thống!")

    _cache_save(self, fp)


VideoSearchEngine._load_database = _load_database_fast
print("✅ đã ghi đè _load_database (cache + đọc song song).")
print("   Bỏ cell này + restart là quay về bản gốc.")
print("   Muốn dựng lại cache từ đầu: rm -rf", CACHE_WRITE)

In [ ]:
# ===== CELL 5 - dựng engine =====
# Model BGE-M3 + CLIP tải ở đây. Luồng làm nóng JSON của cell 1 vẫn đang chạy nền,
# nên hai khoảng chờ mạng này chồng lên nhau thay vì nối tiếp.
import time

_t = time.time()
engine = VideoSearchEngine(db_path=DATABASE_ROOT_PATH)
TIMING["engine"] = time.time() - _t

print(f"\n⏱️  dựng engine: {TIMING['engine']:.0f}s "
      f"· tổng từ đầu phiên: {time.time() - T_BOOT:.0f}s")

---
## Phần API - chỉ chạy nếu muốn dùng web

Bọc quanh `engine` ở trên, **không sửa gì** trong code search.

In [ ]:
%%writefile frame_resolver.py
"""
Nối mixed_database với keyframe/ảnh của BTC.

Bối cảnh (đã đo trên cả 873 video, xem map.md):
  - original_frame_idx của mixed_database LÀ frame thật trong video gốc
    (timestamp_sec == original_frame_idx / fps, đúng tuyệt đối trên mọi video).
  - Nó lệch tối đa ĐÚNG 1 FRAME so với frame_idx của BTC - cùng tập keyframe,
    khác cách làm tròn. 87.4% trùng khít, còn lại lệch 1.
  - Vì vậy: tra sang keyframe BTC gần nhất là an toàn, và nên NỘP frame_idx của BTC
    thay vì original_frame_idx (miễn phí, và bỏ được rủi ro lệch 1 frame có hệ thống).

Cách dùng:
    r = FrameResolver()
    hit = r.resolve("L21_V001", 711)
    hit.submit_frame_id   -> nộp bài
    hit.keyframe_path     -> ảnh hiển thị
    hit.pts_time          -> ffmpeg -ss (cắt clip TRAKE)
"""

from __future__ import annotations

import glob
import os
from dataclasses import dataclass

import numpy as np
import pandas as pd

import glob as _glob

def _auto(pattern: str, depth_max: int = 6) -> str:
    """Dò thư mục dữ liệu BTC, chịu được việc đổi slug dataset."""
    for d in range(1, depth_max + 1):
        hits = _glob.glob("/kaggle/input/" + "*/" * d + pattern)
        if hits:
            return hits[0]
    return ""

MAP_KEYFRAMES_DIR = _auto("map-keyframes*/map-keyframes") or _auto("map-keyframes")

# Dò Keyframes ĐỘC LẬP thay vì suy từ MAP_KEYFRAMES_DIR - suy chuỗi kiểu đó rất dễ
# cắt lố một cấp thư mục, và khi cắt lố thì /search vẫn chạy còn /image lặng lẽ 404.
_kf_one = _auto("Keyframes_*/keyframes")
KEYFRAMES_GLOB = (os.path.dirname(os.path.dirname(_kf_one)) + "/Keyframes_*/keyframes"
                  if _kf_one else "")

# Lệch quá ngưỡng này nghĩa là giả định "cùng tập keyframe, khác làm tròn" đã vỡ.
# Đo được max=1 trên toàn bộ dữ liệu; để 2 cho có biên an toàn.
MAX_EXPECTED_GAP = 2


@dataclass(frozen=True)
class ResolvedFrame:
    video_id: str
    submit_frame_id: int   # frame_idx của BTC - GIÁ TRỊ ĐEM NỘP
    n: int                 # số thứ tự keyframe, quyết định tên file ảnh
    keyframe_path: str     # ảnh để hiển thị lên web
    pts_time: float        # giây, dùng cho `ffmpeg -ss`
    fps: float
    gap: int               # lệch bao nhiêu frame so với giá trị mixed_database đưa vào


class FrameResolver:
    """Nạp map-keyframes một lần, tra O(log n) mỗi frame."""

    def __init__(self, map_dir: str = MAP_KEYFRAMES_DIR, keyframes_glob: str = KEYFRAMES_GLOB):
        self._frames: dict[str, np.ndarray] = {}
        self._n: dict[str, np.ndarray] = {}
        self._pts: dict[str, np.ndarray] = {}
        self._fps: dict[str, float] = {}
        self._kf_dir: dict[str, str] = {}

        for csv_path in glob.glob(os.path.join(map_dir, "*.csv")):
            video_id = os.path.basename(csv_path)[: -len(".csv")]
            df = pd.read_csv(csv_path).sort_values("frame_idx")
            self._frames[video_id] = df["frame_idx"].astype(int).to_numpy()
            self._n[video_id] = df["n"].astype(int).to_numpy()
            self._pts[video_id] = df["pts_time"].astype(float).to_numpy()
            self._fps[video_id] = float(df["fps"].iloc[0])

        # Keyframes nằm rải ở Keyframes_L21/, Keyframes_L22/... - lập chỉ mục một lần
        for root in glob.glob(keyframes_glob):
            for vdir in glob.glob(os.path.join(root, "*")):
                if os.path.isdir(vdir):
                    self._kf_dir[os.path.basename(vdir)] = vdir

        if not self._frames:
            raise FileNotFoundError(
                f"Không đọc được map-keyframes nào từ {map_dir!r}.\n"
                f"Đã attach dataset chứa map-keyframes chưa?"
            )
        if not self._kf_dir:
            raise FileNotFoundError(
                f"Không thấy thư mục keyframe nào khớp {keyframes_glob!r}.\n"
                f"/search sẽ chạy nhưng /image sẽ 404 -> web không có ảnh."
            )
        print(f"FrameResolver: {len(self._frames)} video map · {len(self._kf_dir)} thư mục keyframe")

    def resolve(self, video_id: str, original_frame_idx: int) -> ResolvedFrame:
        """Tra frame của mixed_database sang keyframe BTC gần nhất."""
        if video_id not in self._frames:
            raise KeyError(f"{video_id}: không có map-keyframes")

        frames = self._frames[video_id]
        pos = int(np.searchsorted(frames, original_frame_idx))

        # chọn bên trái hay bên phải, cái nào gần hơn
        best, best_gap = None, None
        for p in (pos - 1, pos):
            if 0 <= p < len(frames):
                g = abs(int(frames[p]) - int(original_frame_idx))
                if best_gap is None or g < best_gap:
                    best, best_gap = p, g

        n = int(self._n[video_id][best])
        kf_dir = self._kf_dir.get(video_id)

        return ResolvedFrame(
            video_id=video_id,
            submit_frame_id=int(frames[best]),
            n=n,
            keyframe_path=os.path.join(kf_dir, f"{n:03d}.jpg") if kf_dir else "",
            pts_time=float(self._pts[video_id][best]),
            fps=self._fps[video_id],
            gap=int(best_gap),
        )

    def resolve_many(self, pairs: list[tuple[str, int]]) -> list[ResolvedFrame]:
        return [self.resolve(v, f) for v, f in pairs]


if __name__ == "__main__":
    r = FrameResolver()
    print(f"Đã nạp {len(r._frames)} video map, {len(r._kf_dir)} thư mục keyframe\n")

    hit = r.resolve("L21_V001", 711)
    print(hit)
    assert os.path.exists(hit.keyframe_path), f"Không thấy ảnh: {hit.keyframe_path}"
    print(f"\n✅ Ảnh tồn tại: {hit.keyframe_path}")

    # Quét toàn bộ mixed_database: mọi frame phải tra được, và không frame nào lệch bất thường
    import json

    DB_PATH = "/kaggle/working/database"
    worst, total, missing_img = 0, 0, 0

    for json_path in sorted(glob.glob(f"{DB_PATH}/*/*/*.json")):
        video_id = os.path.basename(json_path)[: -len(".json")]
        if os.path.basename(os.path.dirname(json_path)) != video_id:
            continue
        with open(json_path, encoding="utf-8") as f:
            frames = json.load(f)["frames"]
        for fr in frames:
            res = r.resolve(video_id, int(fr["original_frame_idx"]))
            worst = max(worst, res.gap)
            total += 1
            if res.keyframe_path and not os.path.exists(res.keyframe_path):
                missing_img += 1

    print(f"\nĐã tra {total:,} frame - lệch lớn nhất {worst} frame, thiếu ảnh {missing_img}")
    if worst <= MAX_EXPECTED_GAP and missing_img == 0:
        print("✅ Mọi frame tra được sang ảnh BTC. Dùng resolver này ở khắp nơi cần hiển thị/cắt clip.")
    else:
        print("❌ Có bất thường - dừng lại soát trước khi tin resolver.")

In [ ]:
%%writefile api_server.py
"""
API bọc quanh `VideoSearchEngine` GỐC - phơi ra ngoài bằng cloudflared.

KHÔNG sửa một dòng nào trong searcher.py. Nó chỉ:
  1. nhận `engine` bạn đã tạo ở cell 5
  2. gọi `analyze_prompt()` + `engine.search()` y như notebook vẫn làm
  3. tra `original_frame_idx` sang keyframe BTC để lấy ảnh (FrameResolver)
  4. trả JSON cho web

Vì sao cần FrameResolver: engine gốc trả `original_frame_idx` (vd 711), nhưng file ảnh BTC
đặt tên theo số thứ tự keyframe (`007.jpg`). Hai hệ đánh số khác nhau - đã đo trên cả 873
video, lệch tối đa 1 frame. Không tra thì web hiện ẢNH SAI mà không báo lỗi.
FrameResolver chỉ TRA CỨU, không tham gia tìm kiếm, không đụng tới thứ hạng.
"""

import hashlib
import io
import os
import re
import subprocess
import threading
import time

TEAM_KEY = "aicdeepbyte"   # key chung của đội - đổi chuỗi này là mọi key cũ hết hiệu lực


def make_key(salt: str = TEAM_KEY) -> str:
    """Key CHUNG CỦA ĐỘI - KHÔNG suy từ tài khoản người chạy.

    Vì sao: `aic.verse.id.vn` là MỘT tunnel dùng chung. Ai bật notebook cũng đăng ký
    connector vào đó, và Cloudflare chia tải ngẫu nhiên giữa các connector. Nếu key
    suy từ username thì:
      - hai người cùng bật -> key của A chỉ đúng ~50% request -> 401 nhấp nháy
      - A tắt, B bật       -> cả đội phải đổi key

    Key chung thì ai chạy cũng vậy, và hai người cùng chạy cũng vô hại (cùng dữ liệu,
    cùng index -> kết quả giống hệt nhau).

    Ưu tiên Kaggle Secret `AIC_ACCESS_KEY` nếu muốn đổi mà không sửa code.
    """
    try:
        from kaggle_secrets import UserSecretsClient
        k = UserSecretsClient().get_secret("AIC_ACCESS_KEY")
        if k:
            return k.strip()
    except Exception:
        pass
    return salt


CLOUDFLARED_URL = (
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
)


def build_app(engine, analyze_fn=None, resolver=None, access_key=None):
    """engine: VideoSearchEngine gốc. analyze_fn: analyze_prompt (tuỳ chọn)."""
    from fastapi import FastAPI, HTTPException, Query
    from fastapi.middleware.cors import CORSMiddleware
    from fastapi.responses import Response
    from pydantic import BaseModel
    from typing import Optional
    import glob

    from frame_resolver import FrameResolver

    resolver = resolver or FrameResolver()

    app = FastAPI(title="AIC Search API")
    app.add_middleware(CORSMiddleware, allow_origins=["*"],
                       allow_methods=["*"], allow_headers=["*"])

    if access_key:
        from fastapi.responses import JSONResponse

        @app.middleware("http")
        async def check_key(request, call_next):
            # Ảnh nạp bằng <img src> nên KHÔNG gửi được header -> phải nhận cả ?key=
            got = (request.headers.get("x-aic-key")
                   or request.query_params.get("key"))
            if request.method == "OPTIONS" or got == access_key:
                return await call_next(request)
            # Middleware này chạy NGOÀI CORSMiddleware (đăng ký sau -> bọc ngoài), nên
            # response 401 trả thẳng từ đây KHÔNG được CORS gắn header. Trình duyệt khi
            # đó chặn luôn và báo "Failed to fetch" thay vì "401" -> người dùng tưởng
            # backend chết, thật ra chỉ sai key. Phải tự gắn header.
            return JSONResponse(
                {"detail": "Sai key hoặc thiếu key"},
                status_code=401,
                headers={
                    "Access-Control-Allow-Origin": request.headers.get("origin", "*"),
                    "Access-Control-Allow-Headers": "*",
                    "Access-Control-Allow-Methods": "*",
                },
            )

    # (video_id, original_frame_idx) -> metadata, để lấy caption/speech/ocr làm bằng chứng
    meta_by_id = {(m["video_id"], int(m["original_frame_idx"])): m for m in engine.metadata}

    video_path = {}
    for d in glob.glob(os.path.dirname(resolver._kf_dir and list(resolver._kf_dir.values())[0] or "")
                       .rsplit("/Keyframes_", 1)[0] + "/Videos_*/video") if resolver._kf_dir else []:
        for f in glob.glob(os.path.join(d, "*.mp4")):
            video_path[os.path.basename(f)[:-4]] = f

    class SearchReq(BaseModel):
        prompt: str = ""
        spatial_context: list = []
        asr_text: list = []
        ocr_text: list = []
        top_k: int = 100
        weights: Optional[dict] = None

    @app.get("/health")
    def health():
        return {"ok": True, "frames": len(engine.metadata),
                "videos": len({m["video_id"] for m in engine.metadata}),
                "llm": analyze_fn is not None}

    @app.post("/search")
    def search(req: SearchReq):
        """Định tuyến theo task, y như process_queries() trong searcher.py.

        Chữ ký engine đã ĐỔI ở bản mới:
          search(action_query=..)            <- một action, cho Task 1/2
          search_temporal_events(actions=..) <- chuỗi action, cho Task 3 (quy hoạch động)
        Cả hai trả về list các LIST id, không phải list chuỗi phẳng như bản cũ.
        """
        w = req.weights or {"visual": 1.0, "speech": 1.0, "ocr": 1.0}

        if req.prompt.strip() and analyze_fn is not None:
            parsed = analyze_fn(req.prompt)
        else:
            # web tự điền 3 trường -> dựng một action giả cho đúng chữ ký mới
            parsed = {
                "task_type": 1,
                "questions": [],
                "actions": [{
                    "spatial_context": req.spatial_context,
                    "asr_text": req.asr_text,
                    "ocr_text": req.ocr_text,
                }],
            }

        task_type = parsed.get("task_type", 1)
        actions = parsed.get("actions", [])
        questions = parsed.get("questions", [])

        if task_type == 3 and len(actions) > 1:
            raw = engine.search_temporal_events(
                actions=actions, top_k=req.top_k, visualize=False, weights=w)
        else:
            raw = engine.search(
                action_query=actions[0] if actions else {},
                top_k=req.top_k, visualize=False, weights=w)

        # ---- chuỗi đúng định dạng BTC, giống hệt process_queries ----
        formatted = []
        for res in raw:
            if not res:
                continue
            vid = res[0].rsplit("_", 1)[0]
            fids = [x.rsplit("_", 1)[1] for x in res]
            if task_type == 3:
                formatted.append(f"{vid}, " + ", ".join(fids))
            elif task_type == 2:
                q = questions[0] if questions else "Unknown Question"
                formatted.append(f"{vid}, {fids[0]}, {q}")
            else:
                formatted.append(f"{vid}, {fids[0]}")

        # ---- candidate phẳng cho lưới ảnh trên web ----
        out = []
        for gi, res in enumerate(raw):
            for s in (res or []):
                video_id, _, fs = s.rpartition("_")
                orig = int(fs)
                hit = resolver.resolve(video_id, orig)
                m = meta_by_id.get((video_id, orig), {})
                sc = getattr(engine, "last_scores", {}).get(f"{video_id}_{orig}") or {}
                out.append({
                    "rank": len(out) + 1,
                    "score": sc.get("total"),
                    "parts": {k: v for k, v in sc.items() if k != "total"} or None,
                    "group": gi,                      # TRAKE: các frame cùng group = một đáp án
                    "video_id": video_id,
                    "frame_id": hit.submit_frame_id,
                    "orig_frame_idx": orig,
                    "keyframe_n": hit.n,
                    "pts_time": hit.pts_time,
                    "fps": float(hit.fps),
                    "image": f"/image?video_id={video_id}&n={hit.n}",
                    "caption": (m.get("visual_caption") or "")[:300],
                    "speech": (m.get("speech_text") or "")[:200],
                    "ocr": (m.get("ocr_text") or "")[:200],
                })

        return {
            "count": len(out),
            "task_type": task_type,
            "parsed": parsed,
            "formatted": formatted,     # đúng định dạng nộp bài
            "results": out,             # để web dựng lưới ảnh
        }

    # Cache thumbnail trong RAM. Không có nó thì mỗi request phải mở JPEG + resize +
    # encode lại; 100 ảnh cùng lúc là tràn hàng đợi. 3000 thumbnail 512px ~ 100 MB.
    from collections import OrderedDict
    _thumb = OrderedDict()
    _THUMB_MAX = 3000

    class ThumbReq(BaseModel):
        items: list = []
        w: int = 320

    @app.get("/image")
    def image(video_id: str, n: int, w: int = Query(512, ge=64, le=1280)):
        key = (video_id, n, w)
        hit = _thumb.get(key)
        if hit is not None:
            _thumb.move_to_end(key)
            return Response(hit, media_type="image/jpeg",
                            headers={"Cache-Control": "public, max-age=86400"})

        from PIL import Image
        kf_dir = resolver._kf_dir.get(video_id)
        if not kf_dir:
            raise HTTPException(404, f"không có keyframe của {video_id}")
        path = os.path.join(kf_dir, f"{n:03d}.jpg")
        if not os.path.exists(path):
            raise HTTPException(404, f"không thấy {path}")

        im = Image.open(path)
        im.thumbnail((w, w))
        buf = io.BytesIO()
        im.convert("RGB").save(buf, "JPEG", quality=80)
        data = buf.getvalue()

        _thumb[key] = data
        while len(_thumb) > _THUMB_MAX:
            _thumb.popitem(last=False)

        return Response(data, media_type="image/jpeg",
                        headers={"Cache-Control": "public, max-age=86400"})


    @app.post("/thumbs")
    def thumbs(req: ThumbReq):
        """Gộp NHIỀU thumbnail vào MỘT response.

        Vì sao cần: Cloudflare giới hạn số request mỗi giây. Lưới 100 ảnh = 100 request
        -> đo thật 36-47% bị 429, kể cả khi chỉ chạy 6 request song song. Giảm song song
        không cứu được vì đây là giới hạn TẦN SUẤT, không phải giới hạn đồng thời.
        Gộp lại thì 100 ảnh chỉ tốn 1 request.
        """
        import base64
        from PIL import Image

        out = {}
        for it in req.items[:200]:
            video_id, n = it[0], int(it[1])
            key = (video_id, n, req.w)
            data = _thumb.get(key)
            if data is None:
                kf_dir = resolver._kf_dir.get(video_id)
                if not kf_dir:
                    continue
                path = os.path.join(kf_dir, f"{n:03d}.jpg")
                if not os.path.exists(path):
                    continue
                im = Image.open(path)
                im.thumbnail((req.w, req.w))
                buf = io.BytesIO()
                im.convert("RGB").save(buf, "JPEG", quality=78)
                data = buf.getvalue()
                _thumb[key] = data
                while len(_thumb) > _THUMB_MAX:
                    _thumb.popitem(last=False)
            else:
                _thumb.move_to_end(key)
            out[f"{video_id}-{n}"] = base64.b64encode(data).decode()
        return {"thumbs": out}

    def _clip_window(video_id, frame_id, seconds):
        """Mốc clip tính bằng SỐ NGUYÊN FRAME, không đi qua pts_time.

        pts_time trong map-keyframes làm tròn 1 chữ số thập phân: frame 997 @25fps thật
        ra ở 39.88s nhưng CSV ghi 39.9 — lệch 0.02s = NỬA FRAME, đủ gây off-by-one khi
        làm tròn. Đi từ frame_id thì start rơi ĐÚNG biên frame, nên frame ở giây 0 của
        clip đúng bằng first_frame.
        """
        hit = resolver.resolve(video_id, frame_id)
        fps = float(hit.fps)
        n = int(round(seconds * fps))
        half = n // 2
        first = max(0, hit.submit_frame_id - half)
        return hit, fps, n, first, first / fps

    @app.get("/clipinfo")
    def clipinfo(video_id: str, frame_id: int, seconds: float = 5.0):
        hit, fps, n, first, start = _clip_window(video_id, frame_id, seconds)
        return {
            "video_id": video_id, "fps": fps,
            "center_frame": hit.submit_frame_id,
            # n / pts_time / gap: cho tab "Nhảy tới frame" trên web. /image và /thumbs
            # đều nhận `n` chứ không nhận frame_id, nên thiếu n là không lấy được các
            # keyframe lân cận. `gap` để giao diện nói rõ đã bắt về keyframe nào, thay
            # vì âm thầm đổi số dưới tay người dùng.
            "n": hit.n,
            "pts_time": hit.pts_time,
            "gap": hit.gap,
            "start_sec": round(start, 6), "seconds": seconds,
            "first_frame": first, "last_frame": first + n - 1, "n_frames": n,
        }

    @app.get("/keyframes")
    def keyframes(video_id: str, n_from: int = 1, n_to: int = 0):
        """Danh sách keyframe THẬT của một video: n, frame_id, pts_time.

        Cần endpoint riêng vì KHÔNG CÓ cách nào suy frame_id từ n ở phía web, mà
        khoảng cách giữa hai keyframe thì KHÔNG ĐỀU. Ví dụ L30_V078:
            n=31 -> 1848 ; n=32 -> 1893 ; n=33 -> 1917
        tức 45 rồi 24 frame, không phải fps=25. Suy bằng cách cộng dồn fps là sai
        ngay từ ô kế bên và sai tích luỹ càng xa càng nặng - ảnh thumbnail lấy theo
        n thì đúng, còn clip lấy theo frame_id suy ra thì trỏ sang cảnh khác.

        n_to = 0 nghĩa là lấy tới hết.
        """
        if video_id not in resolver._frames:
            raise HTTPException(404, f"không thấy video {video_id}")
        frames = resolver._frames[video_id]
        ns = resolver._n[video_id]
        pts = resolver._pts[video_id]
        out = []
        for i in range(len(frames)):
            n = int(ns[i])
            if n < n_from or (n_to and n > n_to):
                continue
            out.append({"n": n, "frame_id": int(frames[i]), "pts_time": float(pts[i])})
        return {"video_id": video_id, "fps": float(resolver._fps[video_id]),
                "count": len(out), "keyframes": out}

    @app.get("/clip")
    def clip(video_id: str, frame_id: int, seconds: float = 5.0):
        """Clip 5s quanh frame. MÃ HOÁ LẠI, không dùng -c copy.

        `-c copy` nhảy tới keyframe gần nhất nên clip không bắt đầu đúng giây yêu cầu —
        sai lệch có thể vài chục frame. Bộ đếm frame trên web dựa vào mốc bắt đầu để
        quy đổi thời gian -> frame, nên lệch mốc là lệch cả bộ đếm. Với TRAKE (khoảng
        đúng dưới 10 frame) thì hỏng hẳn. Mã hoá lại tốn ~1-2s nhưng chính xác tới frame.
        """
        src = video_path.get(video_id)
        if not src:
            raise HTTPException(404, f"không thấy video {video_id}")

        hit, fps, n, first, start = _clip_window(video_id, frame_id, seconds)
        out = f"/tmp/clip_{video_id}_{first}_{n}.mp4"

        if not os.path.exists(out):
            subprocess.run(
                ["ffmpeg", "-y", "-accurate_seek", "-ss", f"{start:.3f}", "-i", src,
                 "-t", str(seconds), "-c:v", "libx264", "-preset", "veryfast",
                 "-crf", "28", "-c:a", "aac", "-b:a", "96k",   # giữ TIẾNG: -an cắt mất âm thanh
                 "-movflags", "+faststart",
                 "-vsync", "cfr", "-r", f"{fps:g}", out],
                check=True, capture_output=True,
            )
        return Response(
            open(out, "rb").read(), media_type="video/mp4",
            headers={
                "Cache-Control": "public, max-age=3600",
                "X-Clip-Start": f"{start:.3f}",
                "X-Clip-Fps": f"{fps:g}",
                "X-Clip-First-Frame": str(first),
                "Access-Control-Expose-Headers": "X-Clip-Start, X-Clip-Fps, X-Clip-First-Frame",
            },
        )

    return app


def _port_free(port: int) -> bool:
    import socket
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", port)) != 0


def serve(engine, analyze_fn=None, port: int = 8000,
          tunnel_token: str = None, hostname: str = None, access_key: str = None):
    """Chạy API trong thread nền + mở tunnel. Trả về (server, url)."""
    import uvicorn

    subprocess.run(["pkill", "-f", "cloudflared tunnel"], capture_output=True)
    time.sleep(0.5)

    if not _port_free(port):
        raise RuntimeError(
            f"Port {port} đang bị server cũ chiếm. Named tunnel ghim cứng port này trên "
            f"Cloudflare nên không nhảy port được.\n-> Restart kernel rồi chạy lại."
        )

    class NonBlocking(uvicorn.Server):
        def install_signal_handlers(self):
            pass

    access_key = access_key or make_key()
    # endpoint /image là `def` (đồng bộ) nên FastAPI chạy nó trong threadpool.
    # Mặc định 40 luồng -> 100 ảnh cùng lúc là xếp hàng. Nâng lên cho thoáng.
    try:
        import anyio.to_thread
        anyio.to_thread.current_default_thread_limiter().total_tokens = 120
    except Exception:
        pass

    app = build_app(engine, analyze_fn, access_key=access_key)
    # bind 0.0.0.0: trong container, `localhost` mà cloudflared dùng có thể là IPv6 ::1
    server = NonBlocking(uvicorn.Config(app, host="0.0.0.0", port=port, log_level="warning"))
    threading.Thread(target=server.run, daemon=True).start()
    for _ in range(100):
        if server.started:
            break
        time.sleep(0.1)
    if not server.started:
        raise RuntimeError(f"Server không khởi động được ở port {port}.")
    print(f"✅ API chạy nền ở 0.0.0.0:{port}")

    if not os.path.exists("./cloudflared"):
        print("⏳ tải cloudflared...")
        subprocess.run(["wget", "-q", CLOUDFLARED_URL, "-O", "./cloudflared"], check=True)
        os.chmod("./cloudflared", 0o755)

    # ---- named tunnel: hostname cố định ----
    if tunnel_token:
        log = open("/tmp/cloudflared.log", "w")
        subprocess.Popen(["./cloudflared", "tunnel", "run", "--token", tunnel_token],
                         stdout=log, stderr=subprocess.STDOUT)
        time.sleep(6)
        txt = open("/tmp/cloudflared.log").read()
        url = f"https://{hostname}" if hostname else None
        print("\n" + "=" * 62)
        print(f"  URL CỐ ĐỊNH:  {url or '(xem Public Hostname trên Cloudflare)'}")
        print("=" * 62)
        if "Registered tunnel connection" in txt:
            print("\n✅ Tunnel đã kết nối.")
        else:
            print("\n❌ Chưa đăng ký được connector. 30 dòng log cuối:")
            print("\n".join(txt.splitlines()[-30:]) or "(rỗng)")

        print(f"\n  🔑 KEY:  {access_key}")
        print("     Dán vào web. Gửi cho đồng đội để cả đội dùng chung backend này.")
        print(f"\n502 -> Public Hostname phải là HTTP + localhost:{port}")
        return server, url

    # ---- quick tunnel: URL ngẫu nhiên ----
    proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}"],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    pat = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
    url = None
    for _ in range(120):
        line = proc.stdout.readline()
        if not line:
            time.sleep(0.25)
            continue
        m = pat.search(line)
        if m:
            url = m.group(0)
            break
    print("\n" + "=" * 62)
    print(f"  URL CÔNG KHAI:  {url or '❌ không lấy được'}")
    print("=" * 62)
    print(f"\n  🔑 KEY:  {access_key}")
    print("\n⚠️  URL đổi mỗi lần chạy - dán cả URL và key vào web.")
    return server, url

In [ ]:
# ===== CELL 8 - mở API ra internet =====
# fastapi/uvicorn và cloudflared đã lo xong ở cell 0 -> ở đây chỉ còn việc chạy.
import sys, time

_t = time.time()
sys.path.insert(0, "/kaggle/working")
for m in ("api_server", "frame_resolver"):
    sys.modules.pop(m, None)
from api_server import serve

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("CF_TUNNEL_TOKEN")
except Exception:
    token = None
    print("ℹ️  Không có CF_TUNNEL_TOKEN -> dùng quick tunnel (URL đổi mỗi phiên).")

server, url = serve(
    engine,                      # engine tạo ở cell 5
    analyze_fn=analyze_prompt,   # để web gửi prompt thô, LLM tự tách + tự nhận Task
    port=8000,
    tunnel_token=token,
    hostname="aic.verse.id.vn",
)
TIMING["serve"] = time.time() - _t

print("\n" + "=" * 52)
print(f"{'TỔNG THỜI GIAN BẬT SERVER':<30}{time.time() - T_BOOT:>8.0f}s")
print("-" * 52)
for k, v in TIMING.items():
    print(f"  {k:<28}{v:>8.0f}s")
print("=" * 52)

In [ ]:
# ===== CELL 9 - thử một truy vấn (chạy SAU khi server đã lên) =====
# Cell này chỉ để nghiệm thu engine, không nằm trên đường găng bật server.
import json

PROMPT = "tìm cảnh bản tin thời sự, người dẫn chương trình đang nói về châu âu đang chống lại nắng nóng"
WEIGHTS = {"visual": 1.0, "speech": 1.0, "ocr": 1.0}

print("🧠 Đang gọi LLM để phân tích truy vấn...")
parsed_query = analyze_prompt(PROMPT)
print("\n✅ KẾT QUẢ LLM PHÂN TÍCH (JSON):")
print(json.dumps(parsed_query, ensure_ascii=False, indent=2))

# Chữ ký MỚI: search(action_query=...) nhận MỘT action, không phải cả parsed_query.
# Task 1/2 -> actions có 1 phần tử. Task 3 -> dùng search_temporal_events (cell 4d).
task_type = parsed_query.get("task_type", 1)
actions = parsed_query.get("actions", [])

print(f"\n🔍 TÌM KIẾM (task {task_type}, {len(actions)} action)...")
if task_type == 3 and len(actions) > 1:
    raw = engine.search_temporal_events(actions=actions, top_k=10, visualize=True, weights=WEIGHTS)
else:
    raw = engine.search(action_query=actions[0] if actions else {},
                        top_k=10, visualize=True, weights=WEIGHTS)

print(f"\n🎯 {len(raw)} kết quả thô (mỗi phần tử là một LIST id):")
print(raw[:5])

In [ ]:
# ===== CELL — CHẠY CẢ GÓI, IN KẾT QUẢ FINAL SUBMISSION =====
# Ra đúng định dạng: list dict {"loại task", "kết quả"}
import json

QUERIES = [
    {"prompt": "tìm cảnh bản tin thời sự, người dẫn chương trình đang nói về giao thông",
     "weights": {"visual": 1.0, "speech": 1.0, "ocr": 1.0}},

    {"prompt": "trong video đua xe đạp, các tay đua còn cách đích bao nhiêu km?",
     "weights": {"visual": 1.0, "speech": 1.0, "ocr": 1.0}},

    {"prompt": "tìm cảnh cua Việt Nam, sau đó là 3 người trên xuồng, sau đó là cảnh sông",
     "weights": {"visual": 1.0, "speech": 1.0, "ocr": 1.0}},
]

# engine=engine  ->  DÙNG LẠI engine đã tạo ở cell 5.
# Bỏ tham số này thì process_queries tự tạo engine mới, nạp lại 4.2GB -> OOM.
FINAL = process_queries(QUERIES, top_k=5, visualize=False, engine=engine)

print("\n" + "=" * 30 + " KẾT QUẢ FINAL SUBMISSION " + "=" * 30)
print(json.dumps(FINAL, ensure_ascii=False, indent=4))

# Đổi visualize=True để xem log LLM phân tích + bảng điểm từng query.
# top_k=100 khi chạy thật (BTC cho tối đa 100 đáp án mỗi truy vấn).

---
## Phần nộp bài

`aic-workspace.json` → mở trên web sửa thứ hạng.  
`submission.zip` → nộp thẳng lên Codabench.

⚠️ Mỗi gói chỉ nộp **3 lần**, tính **lần cuối cùng** - không phải lần tốt nhất.

In [ ]:
%%writefile submission.py
"""
Ghi file nộp bài đúng chuẩn BTC.

Nguồn: trang `Hướng dẫn nộp bài sơ tuyển` trên Codabench competition 10187
(trích nguyên văn ở .scratch/aic-frame-search/research/03-codabench-official.md).

Module này tồn tại để 4 lỗi chí mạng KHÔNG THỂ xảy ra:
  1. Thiếu thư mục `submission/` trong zip  -> BTC liệt kê là lỗi thường gặp #2, hỏng cả bài nộp
  2. Quá 100 dòng mỗi file
  3. Answer Q&A quá 100 ký tự
  4. TRAKE sai số lượng frame so với số events yêu cầu

Dùng `csv` module chứ không nối chuỗi bằng tay, để dấu phẩy và ngoặc kép trong tiếng Việt
được escape đúng.

Bẫy vận hành KHÔNG kiểm được bằng code, phải tự nhớ:
  - Mỗi gói truy vấn chỉ được nộp TỐI ĐA 3 LẦN, và tính LẦN CUỐI CÙNG (không phải lần tốt nhất).
    Nộp thêm một bản tệ hơn là GHI ĐÈ bản tốt. Nộp sai định dạng vẫn tính 1 lần.
"""

from __future__ import annotations

import csv
import io
import os
import shutil
import zipfile
from dataclasses import dataclass, field

MAX_ROWS = 100
MAX_ANSWER_CHARS = 100
VALID_TASKS = ("kis", "qa", "trake")


class SubmissionError(ValueError):
    """Sai định dạng. Ném sớm, ở máy - thay vì phát hiện sau khi đã tiêu một lượt nộp."""


@dataclass
class Answer:
    """Một dòng trong file CSV - tức MỘT đáp án có thứ hạng.

    KIS/Q&A: frame_ids đúng 1 phần tử. TRAKE: N phần tử theo thứ tự thời gian.
    """

    video_id: str
    frame_ids: list[int]
    answer_text: str | None = None   # chỉ Q&A

    def row(self, task: str) -> list:
        if task == "qa":
            return [self.video_id, self.frame_ids[0], self.answer_text]
        return [self.video_id, *self.frame_ids]


@dataclass
class QueryResult:
    """Kết quả cho một truy vấn = một file CSV.

    query_id lấy nguyên tên file truy vấn BTC phát, bỏ đuôi .txt:
    `query-1-kis.txt` -> query_id = "query-1-kis", file nộp = "query-1-kis.csv".
    Hậu tố quyết định loại task.
    """

    query_id: str
    answers: list[Answer] = field(default_factory=list)
    n_events: int | None = None   # TRAKE: số events truy vấn yêu cầu

    @property
    def task(self) -> str:
        suffix = self.query_id.rsplit("-", 1)[-1].lower()
        if suffix not in VALID_TASKS:
            raise SubmissionError(
                f"{self.query_id!r}: không nhận ra loại task từ hậu tố {suffix!r}. "
                f"Tên query phải kết thúc bằng một trong {VALID_TASKS} (vd 'query-1-kis')."
            )
        return suffix

    def validate(self) -> list[str]:
        """Trả về danh sách lỗi. Rỗng nghĩa là hợp lệ."""
        errs: list[str] = []
        task = self.task

        if len(self.answers) > MAX_ROWS:
            errs.append(f"{len(self.answers)} dòng, vượt giới hạn {MAX_ROWS}")
        if not self.answers:
            errs.append("không có dòng nào - nộp file rỗng là phí một truy vấn")

        for i, a in enumerate(self.answers):
            where = f"dòng {i + 1}"
            if not a.video_id or a.video_id.endswith(".mp4"):
                errs.append(f"{where}: video_id {a.video_id!r} phải KHÔNG có đuôi .mp4")
            if any(not isinstance(f, int) or f < 0 for f in a.frame_ids):
                errs.append(f"{where}: frame_id phải là số nguyên không âm, nhận {a.frame_ids}")

            if task == "trake":
                if self.n_events is None:
                    errs.append("TRAKE nhưng chưa khai báo n_events - không kiểm được số frame")
                    break
                if len(a.frame_ids) != self.n_events:
                    errs.append(
                        f"{where}: có {len(a.frame_ids)} frame, truy vấn yêu cầu đúng {self.n_events} events"
                    )
                if list(a.frame_ids) != sorted(a.frame_ids):
                    errs.append(f"{where}: frame phải theo thứ tự thời gian tăng dần, nhận {a.frame_ids}")
            else:
                if len(a.frame_ids) != 1:
                    errs.append(f"{where}: {task} chỉ được 1 frame, nhận {len(a.frame_ids)}")

            if task == "qa":
                if not a.answer_text:
                    errs.append(f"{where}: Q&A thiếu answer")
                elif len(a.answer_text) > MAX_ANSWER_CHARS:
                    errs.append(
                        f"{where}: answer dài {len(a.answer_text)} ký tự, tối đa {MAX_ANSWER_CHARS}"
                    )
            elif a.answer_text:
                errs.append(f"{where}: {task} không được có answer, nhưng có {a.answer_text!r}")

        return errs

    def to_csv(self) -> str:
        buf = io.StringIO()
        # QUOTE_MINIMAL: chỉ bọc khi cần (dấu phẩy/ngoặc kép/xuống dòng) - BTC chấp nhận cả hai cách
        w = csv.writer(buf, lineterminator="\n", quoting=csv.QUOTE_MINIMAL)
        task = self.task
        for a in self.answers[:MAX_ROWS]:
            w.writerow(a.row(task))
        return buf.getvalue()


def write_submission(results: list[QueryResult], out_zip: str, strict: bool = True) -> str:
    """Ghi ra file .zip đúng cấu trúc BTC: bên trong PHẢI có thư mục `submission/`.

    strict=True (mặc định): có bất kỳ lỗi nào thì dừng, không ghi gì.
    """
    all_errs: dict[str, list[str]] = {}
    for r in results:
        errs = r.validate()
        if errs:
            all_errs[r.query_id] = errs

    if all_errs:
        msg = "\n".join(f"  [{q}]\n" + "\n".join(f"    - {e}" for e in errs)
                        for q, errs in all_errs.items())
        if strict:
            raise SubmissionError(f"Bài nộp không hợp lệ, ĐÃ DỪNG (chưa ghi file):\n{msg}")
        print(f"⚠️  Có lỗi nhưng strict=False, vẫn ghi:\n{msg}")

    staging = os.path.join(os.path.dirname(os.path.abspath(out_zip)), "_staging_submission")
    inner = os.path.join(staging, "submission")   # <-- thư mục BẮT BUỘC
    shutil.rmtree(staging, ignore_errors=True)
    os.makedirs(inner, exist_ok=True)

    for r in results:
        with open(os.path.join(inner, f"{r.query_id}.csv"), "w", encoding="utf-8", newline="") as f:
            f.write(r.to_csv())

    with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as z:
        for name in sorted(os.listdir(inner)):
            z.write(os.path.join(inner, name), arcname=f"submission/{name}")

    shutil.rmtree(staging, ignore_errors=True)

    print(f"✅ {out_zip}")
    for r in results:
        print(f"   submission/{r.query_id}.csv  -  {len(r.answers)} dòng ({r.task})")
    print("\n⚠️  Nhớ: tối đa 3 lần nộp mỗi gói, và tính LẦN CUỐI CÙNG chứ không phải lần tốt nhất.")
    return out_zip


def verify_zip(path: str) -> None:
    """Soi lại file zip đã ghi - kiểm đúng thứ đánh giá của BTC sẽ thấy."""
    with zipfile.ZipFile(path) as z:
        names = z.namelist()
        print(f"{path}: {len(names)} file")
        ok = all(n.startswith("submission/") for n in names)
        print(f"{'✅' if ok else '❌'} mọi file nằm trong submission/")
        for n in names:
            body = z.read(n).decode("utf-8")
            lines = [l for l in body.splitlines() if l.strip()]
            print(f"   {n}: {len(lines)} dòng | dòng đầu: {lines[0] if lines else '(rỗng)'}")


if __name__ == "__main__":
    demo = [
        QueryResult("query-1-kis", [Answer("L01_V028", [25300]), Answer("L00_V000", [1234])]),
        QueryResult("query-3-qa", [Answer("L02_V011", [1200], "Năm người"),
                                   Answer("L03_V005", [2800], "Màu đỏ, rất đẹp")]),
        QueryResult("query-4-trake", [Answer("L10_V001", [1200, 1850, 2100, 2450])], n_events=4),
    ]
    write_submission(demo, "/tmp/demo_submission.zip")
    print()
    verify_zip("/tmp/demo_submission.zip")

In [ ]:
# ===== CELL 10 - xuất kết quả: JSON (sửa trên web) + ZIP (nộp thẳng) =====
import json, sys, time
sys.path.insert(0, "/kaggle/working")
for m in ("submission", "frame_resolver"):
    sys.modules.pop(m, None)

from submission import Answer, QueryResult, write_submission, verify_zip
from frame_resolver import FrameResolver

resolver = FrameResolver()
WEIGHTS = {"visual": 1.0, "speech": 1.0, "ocr": 1.0}

# tên PHẢI khớp file BTC phát (bỏ .txt)
QUERIES = [
    {"id": "query-1-kis", "prompt": PROMPT},
    # {"id": "query-2-qa",    "prompt": "...", "answer": "Năm người"},
    # {"id": "query-3-trake", "prompt": "...", "n_events": 4},
]

meta_by_id = {(m["video_id"], int(m["original_frame_idx"])): m for m in engine.metadata}
results, ws_queries = [], []

for q in QUERIES:
    task = q["id"].rsplit("-", 1)[-1]
    parsed = analyze_prompt(q["prompt"])
    actions = parsed.get("actions", [])

    if parsed.get("task_type") == 3 and len(actions) > 1:
        raw = engine.search_temporal_events(actions=actions, top_k=100, visualize=False, weights=WEIGHTS)
    else:
        raw = engine.search(action_query=actions[0] if actions else {},
                            top_k=100, visualize=False, weights=WEIGHTS)

    answers, cands = [], []
    for gi, res in enumerate(raw):
        if not res:
            continue
        frames, first = [], None
        for s in res:
            video_id, _, fs = s.rpartition("_")
            orig = int(fs)
            hit = resolver.resolve(video_id, orig)
            frames.append(hit)
            first = first or (video_id, orig, hit)
            m = meta_by_id.get((video_id, orig), {})
            cands.append({
                "rank": len(cands) + 1, "group": gi, "video_id": video_id,
                "frame_id": hit.submit_frame_id, "orig_frame_idx": orig,
                "keyframe_n": hit.n, "pts_time": hit.pts_time, "score": 0.0, "parts": {},
                "image": f"/image?video_id={video_id}&n={hit.n}",
                "caption": (m.get("visual_caption") or "")[:300],
                "speech": (m.get("speech_text") or "")[:200],
                "ocr": (m.get("ocr_text") or "")[:200],
            })
        answers.append(Answer(
            video_id=first[0],
            frame_ids=[h.submit_frame_id for h in frames],
            answer_text=q.get("answer") if task == "qa" else None,
        ))

    results.append(QueryResult(query_id=q["id"], answers=answers, n_events=q.get("n_events")))
    ws_queries.append({
        "id": q["id"], "task": task, "brief": q["prompt"],
        "prompt": {
            "spatial_context": (actions[0].get("spatial_context", []) if actions else []),
            "asr_text": (actions[0].get("asr_text", []) if actions else []),
            "ocr_text": (actions[0].get("ocr_text", []) if actions else []),
        },
        "n_events": q.get("n_events"),
        "answers": [{"id": f"{q['id']}-{i}",
                     "frames": [{"video_id": c["video_id"], "frame_id": c["frame_id"],
                                 "keyframe_n": c["keyframe_n"], "pts_time": c["pts_time"]}],
                     **({"text": q["answer"]} if task == "qa" else {})}
                    for i, c in enumerate(cands)],
        "candidates": cands, "searchedAt": int(time.time() * 1000), "parsed": parsed,
    })
    print(f"{q['id']:16} {len(answers):3} đáp án   task={task}")

JSON_PATH = "/kaggle/working/aic-workspace.json"
with open(JSON_PATH, "w", encoding="utf-8") as f:
    json.dump({"version": 1, "savedAt": int(time.time() * 1000), "queries": ws_queries},
              f, ensure_ascii=False, indent=2)
print(f"\n✅ {JSON_PATH}")

write_submission(results, "/kaggle/working/submission.zip")
verify_zip("/kaggle/working/submission.zip")

---
## Phụ lục - chẩn đoán bộ nhớ

Không cần chạy để bật server. Dùng khi nghi ngờ OOM.

In [ ]:
# ===== CELL 0 - đo hạn mức bộ nhớ THẬT của container =====
# psutil.virtual_memory().total báo RAM MÁY CHỦ, không phải hạn mức container -> sai.
# Hạn mức thật nằm ở cgroup.
import os

def cgroup_limit_gb():
    for p in ("/sys/fs/cgroup/memory.max",                      # cgroup v2
              "/sys/fs/cgroup/memory/memory.limit_in_bytes"):   # cgroup v1
        try:
            v = open(p).read().strip()
            if v not in ("max", ""):
                return int(v) / 1e9
        except OSError:
            pass
    return None

def cgroup_used_gb():
    for p in ("/sys/fs/cgroup/memory.current",
              "/sys/fs/cgroup/memory/memory.usage_in_bytes"):
        try:
            return int(open(p).read().strip()) / 1e9
        except OSError:
            pass
    return None

import psutil
lim, used = cgroup_limit_gb(), cgroup_used_gb()
print(f"psutil báo (RAM máy chủ, KHÔNG đáng tin): {psutil.virtual_memory().total/1e9:.1f} GB")
print(f"HẠN MỨC THẬT của container            : {lim:.1f} GB" if lim else "không đọc được cgroup")
print(f"đang dùng                             : {used:.1f} GB" if used else "")

import shutil
d = shutil.disk_usage("/kaggle/working")
print(f"\n/kaggle/working: {d.used/1e9:.1f} GB đã dùng / {d.total/1e9:.1f} GB")

# 24.8GB "đang dùng" gồm những gì? anon = bộ nhớ THẬT, file = page cache (thu hồi được)
print("\n--- phân rã bộ nhớ ---")
try:
    st = dict(l.split()[:2] for l in open("/sys/fs/cgroup/memory.stat"))
    anon = int(st.get("anon", 0)) / 1e9
    file_ = int(st.get("file", 0)) / 1e9
    print(f"anon (bộ nhớ THẬT, không thu hồi được): {anon:6.1f} GB")
    print(f"file (page cache, THU HỒI ĐƯỢC)      : {file_:6.1f} GB")
    if lim:
        print(f"\n=> chỗ thật sự còn dùng được: ~{lim - anon:.1f} GB")
    if file_ > 5:
        print("\n⚠️  page cache lớn - chủ yếu do copy/đọc dữ liệu ở /kaggle/working.")
        print("   Nó thu hồi được, nhưng khi cấp phát dồn dập thì kernel có thể không kịp.")
        print("   -> Cân nhắc BỎ cell 4b (copy cục bộ), đọc thẳng từ /kaggle/input.")
except OSError:
    print("(không đọc được memory.stat)")